> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
from pathlib import Path
import os
import subprocess

ROOT = Path("/content/TFM_Rendimiento_cultivos")

# Si el notebook se abre directamente desde GitHub en Colab,
# descargamos automáticamente el repositorio.
if not ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ritaridh/tfm-crop-yield-prediction.git",
            str(ROOT),
        ],
        check=True,
    )

os.chdir(ROOT)

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)

print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PASO 1 - IMPORTAR LIBRERÍAS Y MONTAR GOOGLE DRIVE
# ============================================================


import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Entorno preparado correctamente.")

# Exploración del Dataset creado

In [ ]:
# ============================================================
# PASO 2 - CARGAR DATASET MAESTRO
# ============================================================

RUTA_BASE = "/content/TFM_Rendimiento_cultivos"

ruta_dataset = os.path.join(
    RUTA_BASE,
    "02_Datos_procesados",
    "dataset_maestro_TFM_2017_2023.csv"
)

print("Archivo:", ruta_dataset)
print("Existe:", os.path.exists(ruta_dataset))

df = pd.read_csv(ruta_dataset)

print("\nDimensiones:", df.shape)
print("\nPrimeras filas:")
display(df.head())

In [ ]:
# ============================================================
# PASO 3 - CONTROL DE INTEGRIDAD DEL DATASET
# ============================================================

print("=" * 70)
print("CONTROL DE INTEGRIDAD")
print("=" * 70)

print("\nDimensiones:")
print(df.shape)

print("\nAños:")
print(sorted(df["ano"].unique()))

print("\nCultivos:")
print(df["cultivo"].value_counts())

print("\nProvincias únicas:")
print(df["provincia_estandar"].nunique())

print("\nDuplicados provincia × año × cultivo:")
duplicados = df.duplicated(
    subset=["provincia_estandar", "ano", "cultivo"]
).sum()
print(duplicados)

print("\nValores nulos totales por variable:")
nulos = df.isna().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

print("\nTarget:")
print(df["rendimiento_kg_ha"].describe())

print("\nTarget nulo:")
print(df["rendimiento_kg_ha"].isna().sum())

print("\nTarget <= 0:")
print((df["rendimiento_kg_ha"] <= 0).sum())

In [ ]:
# ============================================================
# PASO 4 - ESTADÍSTICA DESCRIPTIVA DEL TARGET POR CULTIVO
# ============================================================

resumen_target = (
    df.groupby("cultivo")["rendimiento_kg_ha"]
    .describe()
    .round(2)
)

print("ESTADÍSTICA DESCRIPTIVA DEL RENDIMIENTO POR CULTIVO")
display(resumen_target)

# Estadísticos adicionales
estadisticos_extra = (
    df.groupby("cultivo")["rendimiento_kg_ha"]
    .agg(
        mediana="median",
        varianza="var",
        asimetria="skew"
    )
    .round(3)
)

print("\nESTADÍSTICOS ADICIONALES")
display(estadisticos_extra)

In [ ]:
# ============================================================
# PASO 5 - DISTRIBUCIÓN DEL RENDIMIENTO POR CULTIVO
# ============================================================

import matplotlib.pyplot as plt

cultivos = [
    ("barley", "Cebada"),
    ("common_soft_wheat", "Trigo blando")
]

for cultivo, etiqueta in cultivos:

    datos = df.loc[
        df["cultivo"] == cultivo,
        "rendimiento_kg_ha"
    ]

    plt.figure(figsize=(8, 5))

    plt.hist(
        datos,
        bins=20,
        edgecolor="black"
    )

    plt.axvline(
        datos.mean(),
        linestyle="--",
        label=f"Media = {datos.mean():.0f}"
    )

    plt.axvline(
        datos.median(),
        linestyle=":",
        label=f"Mediana = {datos.median():.0f}"
    )

    plt.xlabel("Rendimiento (kg/ha)")
    plt.ylabel("Frecuencia")
    plt.title(
        f"Distribución del rendimiento - {etiqueta}"
    )

    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
# ============================================================
# PASO 6 - VALORES EXTREMOS DEL TARGET
# ============================================================

for cultivo, etiqueta in cultivos:

    datos = df.loc[
        df["cultivo"] == cultivo,
        "rendimiento_kg_ha"
    ]

    plt.figure(figsize=(8, 3))

    plt.boxplot(
        datos,
        vert=False
    )

    plt.xlabel("Rendimiento (kg/ha)")
    plt.title(
        f"Boxplot del rendimiento - {etiqueta}"
    )

    plt.grid(alpha=0.2)
    plt.show()


# ------------------------------------------------------------
# Identificación mediante IQR
# ------------------------------------------------------------

outliers_target = []

for cultivo, grupo in df.groupby("cultivo"):

    q1 = grupo["rendimiento_kg_ha"].quantile(0.25)
    q3 = grupo["rendimiento_kg_ha"].quantile(0.75)

    iqr = q3 - q1

    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    extremos = grupo[
        (grupo["rendimiento_kg_ha"] < limite_inf) |
        (grupo["rendimiento_kg_ha"] > limite_sup)
    ].copy()

    extremos["limite_inferior_IQR"] = limite_inf
    extremos["limite_superior_IQR"] = limite_sup

    outliers_target.append(extremos)

outliers_target = pd.concat(
    outliers_target,
    ignore_index=True
)

print(
    "Número de observaciones identificadas "
    "por criterio IQR:",
    len(outliers_target)
)

display(
    outliers_target[
        [
            "ano",
            "provincia_estandar",
            "cultivo",
            "rendimiento_kg_ha",
            "limite_inferior_IQR",
            "limite_superior_IQR"
        ]
    ].sort_values(
        ["cultivo", "rendimiento_kg_ha"]
    )
)

In [ ]:
# ============================================================
# PASO 7 - EVOLUCIÓN TEMPORAL DEL RENDIMIENTO
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

resumen_anual = (
    df.groupby(["ano", "cultivo"])["rendimiento_kg_ha"]
    .agg(
        n="count",
        media="mean",
        mediana="median",
        std="std",
        minimo="min",
        maximo="max"
    )
    .round(2)
    .reset_index()
)

display(resumen_anual)

for cultivo, etiqueta in [
    ("barley", "Cebada"),
    ("common_soft_wheat", "Trigo blando")
]:

    temp = resumen_anual[
        resumen_anual["cultivo"] == cultivo
    ]

    plt.figure(figsize=(9, 5))

    plt.plot(
        temp["ano"],
        temp["media"],
        marker="o",
        label="Media"
    )

    plt.plot(
        temp["ano"],
        temp["mediana"],
        marker="s",
        linestyle="--",
        label="Mediana"
    )

    plt.xlabel("Campaña")
    plt.ylabel("Rendimiento (kg/ha)")
    plt.title(f"Evolución temporal del rendimiento - {etiqueta}")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

In [ ]:
# ============================================================
# PASO 8 - VARIABILIDAD PROVINCIAL DEL RENDIMIENTO
# ============================================================

resumen_provincia = (
    df.groupby(["cultivo", "provincia_estandar"])["rendimiento_kg_ha"]
    .agg(
        n="count",
        media="mean",
        mediana="median",
        std="std",
        minimo="min",
        maximo="max"
    )
    .round(2)
    .reset_index()
)

for cultivo, etiqueta in [
    ("barley", "Cebada"),
    ("common_soft_wheat", "Trigo blando")
]:

    temp = (
        resumen_provincia[
            resumen_provincia["cultivo"] == cultivo
        ]
        .sort_values("media")
    )

    print("\n", "=" * 60)
    print(etiqueta.upper())
    print("=" * 60)

    display(temp)

    plt.figure(figsize=(10, 12))

    plt.barh(
        temp["provincia_estandar"],
        temp["media"]
    )

    plt.xlabel("Rendimiento medio 2017-2023 (kg/ha)")
    plt.ylabel("Provincia")
    plt.title(
        f"Rendimiento medio provincial - {etiqueta}"
    )

    plt.grid(axis="x", alpha=0.2)
    plt.show()

In [ ]:
# ============================================================
# PASO 9 - MAPA DE CALOR PROVINCIA × AÑO DEL RENDIMIENTO
# ============================================================

import matplotlib.pyplot as plt

for cultivo, etiqueta in [
    ("barley", "Cebada"),
    ("common_soft_wheat", "Trigo blando")
]:

    matriz = (
        df[df["cultivo"] == cultivo]
        .pivot(
            index="provincia_estandar",
            columns="ano",
            values="rendimiento_kg_ha"
        )
    )

    # Ordenar provincias por rendimiento medio
    orden = matriz.mean(axis=1).sort_values().index
    matriz = matriz.loc[orden]

    plt.figure(figsize=(10, 13))

    im = plt.imshow(
        matriz,
        aspect="auto"
    )

    plt.colorbar(
        im,
        label="Rendimiento (kg/ha)"
    )

    plt.xticks(
        range(len(matriz.columns)),
        matriz.columns
    )

    plt.yticks(
        range(len(matriz.index)),
        matriz.index
    )

    plt.xlabel("Campaña")
    plt.ylabel("Provincia")

    plt.title(
        f"Mapa de calor del rendimiento - {etiqueta}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PASO 10 - ESTADÍSTICA DESCRIPTIVA DE PREDICTORES
# ============================================================

excluir = [
    "ano",
    "cultivo",
    "provincia_estandar",
    "rendimiento_kg_ha"
]

predictores = [
    c for c in df.columns
    if c not in excluir
]

resumen_predictores = (
    df[predictores]
    .describe()
    .T
)

resumen_predictores["mediana"] = (
    df[predictores].median()
)

resumen_predictores["asimetria"] = (
    df[predictores].skew()
)

resumen_predictores["n_nulos"] = (
    df[predictores].isna().sum()
)

resumen_predictores["pct_nulos"] = (
    df[predictores].isna().mean() * 100
)

resumen_predictores = resumen_predictores[
    [
        "count",
        "mean",
        "std",
        "min",
        "25%",
        "mediana",
        "75%",
        "max",
        "asimetria",
        "n_nulos",
        "pct_nulos"
    ]
].round(3)

display(resumen_predictores)

In [ ]:
# ============================================================
# PASO 11 - CORRELACIÓN CON EL TARGET
# PEARSON Y SPEARMAN
# ============================================================

predictores_num = [
    c for c in df.columns
    if c not in [
        "ano",
        "cultivo",
        "provincia_estandar",
        "rendimiento_kg_ha"
    ]
]

corr_pearson = (
    df[predictores_num + ["rendimiento_kg_ha"]]
    .corr(method="pearson")["rendimiento_kg_ha"]
    .drop("rendimiento_kg_ha")
    .sort_values(key=abs, ascending=False)
)

corr_spearman = (
    df[predictores_num + ["rendimiento_kg_ha"]]
    .corr(method="spearman")["rendimiento_kg_ha"]
    .drop("rendimiento_kg_ha")
    .sort_values(key=abs, ascending=False)
)

tabla_corr_target = pd.DataFrame({
    "Pearson": corr_pearson,
    "Spearman": corr_spearman
})

display(tabla_corr_target)

In [ ]:
# ============================================================
# PASO 12 - CORRELACIONES CON TARGET POR CULTIVO
# ============================================================

for cultivo, etiqueta in [
    ("barley", "Cebada"),
    ("common_soft_wheat", "Trigo blando")
]:

    df_c = df[df["cultivo"] == cultivo]

    pearson = (
        df_c[predictores_num + ["rendimiento_kg_ha"]]
        .corr(method="pearson")["rendimiento_kg_ha"]
        .drop("rendimiento_kg_ha")
        .sort_values(key=abs, ascending=False)
    )

    spearman = (
        df_c[predictores_num + ["rendimiento_kg_ha"]]
        .corr(method="spearman")["rendimiento_kg_ha"]
        .drop("rendimiento_kg_ha")
        .sort_values(key=abs, ascending=False)
    )

    tabla = pd.DataFrame({
        "Pearson": pearson,
        "Spearman": spearman
    })

    print("\n" + "=" * 70)
    print(etiqueta.upper())
    print("=" * 70)

    display(tabla.head(20))

In [ ]:
# ============================================================
# PASO 13 - HEATMAP ERA5
# ============================================================

cols_era5 = [
    c for c in df.columns
    if (
        c.startswith("temp_")
        or c.startswith("precipitacion_")
        or c.startswith("radiacion_")
    )
]

corr_era5 = df[cols_era5].corr()

plt.figure(figsize=(14, 11))

im = plt.imshow(
    corr_era5,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(im, label="Correlación Pearson")

plt.xticks(
    range(len(cols_era5)),
    cols_era5,
    rotation=90
)

plt.yticks(
    range(len(cols_era5)),
    cols_era5
)

plt.title("Matriz de correlación - ERA5-Land")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 14 - HEATMAP SENTINEL-2
# ============================================================

cols_s2 = [
    c for c in df.columns
    if c.startswith(("NDVI_", "EVI_", "NDMI_"))
]

corr_s2 = df[cols_s2].corr()

plt.figure(figsize=(12, 10))

im = plt.imshow(
    corr_s2,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(im, label="Correlación Pearson")

plt.xticks(
    range(len(cols_s2)),
    cols_s2,
    rotation=90
)

plt.yticks(
    range(len(cols_s2)),
    cols_s2
)

plt.title("Matriz de correlación - Sentinel-2")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 15 - HEATMAP SOILGRIDS + EUROCROPS
# ============================================================

cols_static = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm",
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha"
]

corr_static = df[cols_static].corr()

plt.figure(figsize=(9, 8))

im = plt.imshow(
    corr_static,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(im, label="Correlación Pearson")

plt.xticks(
    range(len(cols_static)),
    cols_static,
    rotation=90
)

plt.yticks(
    range(len(cols_static)),
    cols_static
)

plt.title("Matriz de correlación - SoilGrids + EuroCrops")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 16 - PARES DE PREDICTORES ALTAMENTE CORRELACIONADOS
# ============================================================

corr_pred = df[predictores_num].corr().abs()

upper = corr_pred.where(
    np.triu(
        np.ones(corr_pred.shape),
        k=1
    ).astype(bool)
)

pares_altos = []

for col in upper.columns:
    for idx in upper.index:
        valor = upper.loc[idx, col]

        if pd.notna(valor) and valor >= 0.85:
            pares_altos.append({
                "variable_1": idx,
                "variable_2": col,
                "correlacion_abs": valor
            })

pares_altos_df = pd.DataFrame(
    pares_altos
).sort_values(
    "correlacion_abs",
    ascending=False
)

print(
    "Número de pares con |r| >= 0.85:",
    len(pares_altos_df)
)

display(pares_altos_df)

In [ ]:
# ============================================================
# PASO 17 - VIF
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

pred_vif = [
    c for c in predictores_num
    if df[c].isna().sum() == 0
]

X_vif = df[pred_vif].copy()

# Estandarización solo para estabilidad numérica del diagnóstico
X_vif = (X_vif - X_vif.mean()) / X_vif.std()

vif_df = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
}).sort_values("VIF", ascending=False)

display(vif_df)

In [ ]:
# ============================================================
# PASO 18 - MISSINGNESS
# ============================================================

cols_na = [
    c for c in df.columns
    if df[c].isna().sum() > 0
]

missing_summary = pd.DataFrame({
    "variable": cols_na,
    "n_nulos": [df[c].isna().sum() for c in cols_na],
    "pct_nulos": [
        round(df[c].isna().mean() * 100, 2)
        for c in cols_na
    ]
})

display(missing_summary)

# Patrón de combinaciones de NaN
patrones_na = (
    df[cols_na]
    .isna()
    .astype(int)
    .value_counts()
    .reset_index(name="n_observaciones")
)

display(patrones_na)

In [ ]:
# ============================================================
# PASO 19 - REDUNDANCIA ERA5
# ============================================================

cols_era5_redundancia = [
    "temp_media_c_campana",
    "temp_media_c_ene_feb",
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "temp_media_c_jul_ago",
    "temp_media_c_sep_dic",
    "precipitacion_mm_campana",
    "precipitacion_mm_ene_feb",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "precipitacion_mm_jul_ago",
    "precipitacion_mm_sep_dic",
    "radiacion_mj_m2_campana",
    "radiacion_mj_m2_ene_feb",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_jul_ago",
    "radiacion_mj_m2_sep_dic"
]

corr_redundancia = df[cols_era5_redundancia].corr()

plt.figure(figsize=(13, 11))
im = plt.imshow(corr_redundancia, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, label="Correlación Pearson")
plt.xticks(
    range(len(cols_era5_redundancia)),
    cols_era5_redundancia,
    rotation=90
)
plt.yticks(
    range(len(cols_era5_redundancia)),
    cols_era5_redundancia
)
plt.title("Redundancia entre variables ERA5")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 20 - REDUNDANCIA SENTINEL-2 POR VENTANA
# ============================================================

ventanas_s2 = [
    "nov_dec",
    "jan_feb",
    "mar_apr",
    "may_jun",
    "jul"
]

resultados_s2_corr = []

for ventana in ventanas_s2:

    cols = [
        f"NDVI_{ventana}",
        f"EVI_{ventana}",
        f"NDMI_{ventana}"
    ]

    corr = df[cols].corr()

    print("\n" + "=" * 60)
    print("VENTANA:", ventana)
    print("=" * 60)
    display(corr)

    resultados_s2_corr.append(corr)

In [ ]:
# ============================================================
# PASO 21 - VIF SIN VARIABLES AGREGADAS DE CAMPAÑA
# ============================================================

variables_campana = [
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]

pred_vif_reducido = [
    c for c in predictores_num
    if (
        c not in variables_campana
        and df[c].isna().sum() == 0
    )
]

X_vif2 = df[pred_vif_reducido].copy()

X_vif2 = (
    X_vif2 - X_vif2.mean()
) / X_vif2.std()

vif_reducido = pd.DataFrame({
    "variable": X_vif2.columns,
    "VIF": [
        variance_inflation_factor(
            X_vif2.values,
            i
        )
        for i in range(X_vif2.shape[1])
    ]
}).sort_values(
    "VIF",
    ascending=False
)

print(
    "Variables con VIF > 10:",
    (vif_reducido["VIF"] > 10).sum()
)

print(
    "Variables con VIF > 5:",
    (vif_reducido["VIF"] > 5).sum()
)

display(vif_reducido)

In [ ]:
# ============================================================
# PASO 22 - HEATMAP GLOBAL DE PREDICTORES
# ============================================================

corr_global = df[predictores_num].corr()

plt.figure(figsize=(18, 16))

im = plt.imshow(
    corr_global,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    im,
    label="Correlación Pearson"
)

plt.xticks(
    range(len(predictores_num)),
    predictores_num,
    rotation=90,
    fontsize=7
)

plt.yticks(
    range(len(predictores_num)),
    predictores_num,
    fontsize=7
)

plt.title(
    "Matriz global de correlaciones entre predictores"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 23 - CORRELACIÓN PREDICTORES VS RENDIMIENTO
# ============================================================

corr_target_plot = (
    df[predictores_num + ["rendimiento_kg_ha"]]
    .corr(method="spearman")["rendimiento_kg_ha"]
    .drop("rendimiento_kg_ha")
    .sort_values()
)

plt.figure(figsize=(10, 12))

plt.barh(
    corr_target_plot.index,
    corr_target_plot.values
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel(
    "Correlación de Spearman con rendimiento"
)

plt.ylabel("Predictor")

plt.title(
    "Asociación de los predictores con el rendimiento"
)

plt.grid(
    axis="x",
    alpha=0.2
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASO 24 - DISTRIBUCIONES DE PREDICTORES
# ============================================================

for variable in predictores_num:

    datos = df[variable].dropna()

    plt.figure(figsize=(7, 4))

    plt.hist(
        datos,
        bins=25,
        edgecolor="black"
    )

    plt.axvline(
        datos.mean(),
        linestyle="--",
        label=f"Media = {datos.mean():.2f}"
    )

    plt.axvline(
        datos.median(),
        linestyle=":",
        label=f"Mediana = {datos.median():.2f}"
    )

    plt.xlabel(variable)
    plt.ylabel("Frecuencia")

    plt.title(
        f"Distribución de {variable}"
    )

    plt.legend()
    plt.grid(alpha=0.2)

    plt.show()

In [ ]:
# ============================================================
# PASO 25 - DETECCIÓN DE OUTLIERS EN PREDICTORES
# ============================================================

resultados_outliers = []

for variable in predictores_num:

    datos = df[variable].dropna()

    q1 = datos.quantile(0.25)
    q3 = datos.quantile(0.75)
    iqr = q3 - q1

    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    mask = (
        (df[variable] < limite_inf) |
        (df[variable] > limite_sup)
    )

    n_outliers = mask.sum()

    resultados_outliers.append({
        "variable": variable,
        "n_outliers": n_outliers,
        "pct_outliers": round(
            n_outliers / df[variable].notna().sum() * 100,
            2
        ),
        "min": datos.min(),
        "max": datos.max(),
        "limite_IQR_inf": limite_inf,
        "limite_IQR_sup": limite_sup
    })

df_outliers = (
    pd.DataFrame(resultados_outliers)
    .sort_values("pct_outliers", ascending=False)
)

display(df_outliers)

print("\nVariables con algún outlier:",
      (df_outliers["n_outliers"] > 0).sum())

print("\n10 variables con mayor porcentaje de outliers:")
display(df_outliers.head(10))

In [ ]:
# ============================================================
# PASO 26 - RELACIÓN PREDICTORES PRINCIPALES VS RENDIMIENTO
# ============================================================

variables_principales = [
    "NDVI_may_jun",
    "NDMI_may_jun",
    "NDVI_mar_apr",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "temp_media_c_mar_abr"
]

for variable in variables_principales:

    for cultivo in ["barley", "common_soft_wheat"]:

        datos = df[
            df["cultivo"] == cultivo
        ][[variable, "rendimiento_kg_ha"]].dropna()

        plt.figure(figsize=(7, 5))

        plt.scatter(
            datos[variable],
            datos["rendimiento_kg_ha"],
            alpha=0.6
        )

        # Tendencia lineal solo como ayuda visual
        z = np.polyfit(
            datos[variable],
            datos["rendimiento_kg_ha"],
            1
        )

        p = np.poly1d(z)

        x_line = np.linspace(
            datos[variable].min(),
            datos[variable].max(),
            100
        )

        plt.plot(
            x_line,
            p(x_line),
            linewidth=2
        )

        plt.xlabel(variable)
        plt.ylabel("Rendimiento (kg/ha)")
        plt.title(f"{variable} vs rendimiento — {cultivo}")
        plt.grid(alpha=0.2)

        plt.show()

In [ ]:
# ============================================================
# PASO 27 - VARIABILIDAD INTERANUAL DEL RENDIMIENTO
# ============================================================

rendimiento_anual = (
    df.groupby(["ano", "cultivo"])["rendimiento_kg_ha"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(rendimiento_anual)

for cultivo in ["barley", "common_soft_wheat"]:

    datos = rendimiento_anual[
        rendimiento_anual["cultivo"] == cultivo
    ]

    plt.figure(figsize=(8, 5))

    plt.plot(
        datos["ano"],
        datos["mean"],
        marker="o",
        label="Media"
    )

    plt.plot(
        datos["ano"],
        datos["median"],
        marker="o",
        label="Mediana"
    )

    plt.xlabel("Año")
    plt.ylabel("Rendimiento (kg/ha)")
    plt.title(f"Evolución interanual del rendimiento — {cultivo}")
    plt.legend()
    plt.grid(alpha=0.2)

    plt.show()

In [ ]:
# ============================================================
# PASO 28 - EVOLUCIÓN INTERANUAL DE PREDICTORES CLAVE
# ============================================================

variables_temporales = [
    "NDVI_mar_apr",
    "NDVI_may_jun",
    "NDMI_may_jun",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "temp_media_c_mar_abr"
]

for variable in variables_temporales:

    resumen = (
        df.groupby(["ano", "cultivo"])[variable]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(8, 5))

    for cultivo in ["barley", "common_soft_wheat"]:

        datos = resumen[
            resumen["cultivo"] == cultivo
        ]

        plt.plot(
            datos["ano"],
            datos[variable],
            marker="o",
            label=cultivo
        )

    plt.xlabel("Año")
    plt.ylabel(variable)
    plt.title(f"Evolución interanual — {variable}")
    plt.legend()
    plt.grid(alpha=0.2)

    plt.show()

In [ ]:
# ============================================================
# PASO 29 - COMPARACIÓN DEL RENDIMIENTO ENTRE CULTIVOS
# ============================================================

from scipy import stats

barley = df.loc[
    df["cultivo"] == "barley",
    "rendimiento_kg_ha"
].dropna()

wheat = df.loc[
    df["cultivo"] == "common_soft_wheat",
    "rendimiento_kg_ha"
].dropna()

print("Cebada n =", len(barley))
print("Trigo n =", len(wheat))

# Normalidad
shapiro_barley = stats.shapiro(barley)
shapiro_wheat = stats.shapiro(wheat)

print("\nShapiro-Wilk")
print("Cebada:", shapiro_barley)
print("Trigo:", shapiro_wheat)

# Igualdad de varianzas
levene = stats.levene(
    barley,
    wheat,
    center="median"
)

print("\nLevene:")
print(levene)

# Comparación paramétrica
ttest = stats.ttest_ind(
    barley,
    wheat,
    equal_var=False
)

print("\nWelch t-test:")
print(ttest)

# Comparación no paramétrica
mannwhitney = stats.mannwhitneyu(
    barley,
    wheat,
    alternative="two-sided"
)

print("\nMann-Whitney U:")
print(mannwhitney)

In [ ]:
# ============================================================
# PASO 30 - TAMAÑO DEL EFECTO
# ============================================================

mean_diff = wheat.mean() - barley.mean()

pooled_std = np.sqrt(
    (
        (len(barley)-1)*barley.var()
        +
        (len(wheat)-1)*wheat.var()
    )
    /
    (len(barley)+len(wheat)-2)
)

cohen_d = mean_diff / pooled_std

print(
    "Diferencia de medias trigo - cebada:",
    round(mean_diff, 2),
    "kg/ha"
)

print(
    "Cohen's d:",
    round(cohen_d, 3)
)

In [ ]:
# ============================================================
# PASO 31 - CORRELACIÓN CON EL TARGET POR BLOQUE
# ============================================================

# ------------------------------------------------------------
# Definir columnas de cada fuente de datos
# ------------------------------------------------------------

cols_era5 = [
    c for c in df.columns
    if (
        c.startswith("temp_media_")
        or c.startswith("temp_rocio_")
        or c.startswith("precipitacion_")
        or c.startswith("radiacion_")
    )
]

cols_soilgrids = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

cols_eurocrops = [
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha"
]

cols_sentinel = [
    c for c in df.columns
    if c.startswith(("NDVI_", "EVI_", "NDMI_"))
]

# ------------------------------------------------------------
# Agrupar por bloques
# ------------------------------------------------------------

bloques_modelo = {
    "ERA5": cols_era5,
    "SoilGrids": cols_soilgrids,
    "EuroCrops": cols_eurocrops,
    "Sentinel-2": cols_sentinel
}

# ------------------------------------------------------------
# Calcular correlaciones con rendimiento
# ------------------------------------------------------------

resumen_bloques = []

for bloque, columnas in bloques_modelo.items():

    for metodo in ["pearson", "spearman"]:

        corr = (
            df[columnas + ["rendimiento_kg_ha"]]
            .corr(method=metodo)["rendimiento_kg_ha"]
            .drop("rendimiento_kg_ha")
        )

        resumen_bloques.append({
            "bloque": bloque,
            "metodo": metodo,
            "n_variables": len(columnas),
            "media_abs_correlacion": corr.abs().mean(),
            "max_abs_correlacion": corr.abs().max(),
            "variable_max": corr.abs().idxmax(),
            "correlacion_variable_max": corr.loc[corr.abs().idxmax()]
        })

resumen_bloques_df = pd.DataFrame(resumen_bloques)

display(
    resumen_bloques_df.round(4)
)

In [ ]:
# ============================================================
# PASO 32 - REDUNDANCIA INTERNA POR BLOQUE
# ============================================================

def resumen_redundancia(columnas, nombre):

    corr = df[columnas].corr().abs()

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    valores = upper.stack()

    print("\n" + "=" * 70)
    print(nombre.upper())
    print("=" * 70)

    print("Número de variables:", len(columnas))

    print(
        "Pares |r| >= 0.85:",
        int((valores >= 0.85).sum())
    )

    print(
        "Pares |r| >= 0.90:",
        int((valores >= 0.90).sum())
    )

    print(
        "Pares |r| >= 0.95:",
        int((valores >= 0.95).sum())
    )

    print(
        "Correlación máxima:",
        round(valores.max(), 4)
    )

    print("\n10 pares más correlacionados:")

    top10 = (
        valores
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    top10.columns = [
        "variable_1",
        "variable_2",
        "correlacion_abs"
    ]

    display(top10)


resumen_redundancia(
    cols_era5,
    "ERA5"
)

resumen_redundancia(
    cols_soilgrids,
    "SoilGrids"
)

resumen_redundancia(
    cols_eurocrops,
    "EuroCrops"
)

resumen_redundancia(
    cols_sentinel,
    "Sentinel-2"
)

In [ ]:
# ============================================================
# PASO 33 - ESTRUCTURA ESPACIO-TEMPORAL DEL RENDIMIENTO
# ============================================================

# Variabilidad entre años
resumen_ano = (
    df.groupby("ano")["rendimiento_kg_ha"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

print("=" * 70)
print("RENDIMIENTO POR AÑO")
print("=" * 70)
display(resumen_ano)


# Variabilidad entre provincias
resumen_provincia = (
    df.groupby("provincia_estandar")["rendimiento_kg_ha"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
    .round(2)
)

print("\n" + "=" * 70)
print("RENDIMIENTO POR PROVINCIA")
print("=" * 70)
display(resumen_provincia)


# Año × cultivo
resumen_ano_cultivo = (
    df.groupby(["ano", "cultivo"])["rendimiento_kg_ha"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

print("\n" + "=" * 70)
print("RENDIMIENTO POR AÑO Y CULTIVO")
print("=" * 70)
display(resumen_ano_cultivo)

In [ ]:
# ============================================================
# PASO 34 - EFECTO DE AÑO, PROVINCIA Y CULTIVO
# ============================================================

import statsmodels.formula.api as smf

modelos_exploratorios = {

    "Año":
        "rendimiento_kg_ha ~ C(ano)",

    "Cultivo":
        "rendimiento_kg_ha ~ C(cultivo)",

    "Provincia":
        "rendimiento_kg_ha ~ C(provincia_estandar)",

    "Año + Cultivo":
        "rendimiento_kg_ha ~ C(ano) + C(cultivo)",

    "Año + Provincia + Cultivo":
        "rendimiento_kg_ha ~ C(ano) + C(provincia_estandar) + C(cultivo)"
}

resultados = []

for nombre, formula in modelos_exploratorios.items():

    modelo = smf.ols(
        formula,
        data=df
    ).fit()

    resultados.append({
        "modelo": nombre,
        "R2": modelo.rsquared,
        "R2_ajustado": modelo.rsquared_adj,
        "AIC": modelo.aic,
        "BIC": modelo.bic
    })

resultados_estructura = pd.DataFrame(resultados)

display(
    resultados_estructura.round(4)
)

In [ ]:
# ============================================================
# PASO 35 - MAPA DE CALOR PROVINCIA × AÑO
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

tabla_heatmap = df.pivot_table(
    index="provincia_estandar",
    columns="ano",
    values="rendimiento_kg_ha",
    aggfunc="mean"
)

plt.figure(figsize=(11, 14))

sns.heatmap(
    tabla_heatmap,
    cmap="YlGnBu",
    linewidths=0.2
)

plt.title(
    "Rendimiento medio por provincia y año",
    fontsize=14
)

plt.xlabel("Año")
plt.ylabel("Provincia")

plt.tight_layout()
plt.show()

# Auditoria estructural temporal para deep Learning

In [ ]:
# ============================================================
# PASO 36 - AUDITORÍA DE ESTRUCTURA TEMPORAL PARA DEEP LEARNING
# ============================================================

# Trabajamos sobre el dataset maestro ya cargado
# y examinamos únicamente los predictores finales.

columnas_excluir = [
    "ano",
    "provincia_estandar",
    "rendimiento_kg_ha",
    "cultivo"
]

predictores_numericos = [
    c for c in df.columns
    if c not in columnas_excluir
]

# ------------------------------------------------------------
# 1. Clasificación de variables por fuente
# ------------------------------------------------------------

variables_sentinel = [
    c for c in predictores_numericos
    if c.startswith(("NDVI_", "EVI_", "NDMI_"))
]

variables_era5 = [
    c for c in predictores_numericos
    if c.startswith((
        "temp_media_c_",
        "temp_rocio_c_",
        "precipitacion_mm_",
        "radiacion_mj_m2_"
    ))
]

variables_suelo = [
    c for c in predictores_numericos
    if c in [
        "clay_pct_0_30cm",
        "sand_pct_0_30cm",
        "soc_g_kg_0_30cm",
        "ph_0_30cm",
        "bdod_g_cm3_0_30cm"
    ]
]

variables_eurocrops = [
    c for c in predictores_numericos
    if c in [
        "n_recintos",
        "superficie_eurocrops_ha",
        "superficie_media_recinto_ha"
    ]
]

# ------------------------------------------------------------
# 2. Variables que conservan estructura temporal
# ------------------------------------------------------------

periodos = [
    "jan_feb",
    "ene_feb",
    "mar_apr",
    "mar_abr",
    "may_jun",
    "jul",
    "jul_ago",
    "sep_dic",
    "nov_dec"
]

variables_temporales = [
    c for c in predictores_numericos
    if any(periodo in c for periodo in periodos)
]

variables_campana = [
    c for c in predictores_numericos
    if c.endswith("_campana")
]

# ------------------------------------------------------------
# 3. Mostrar resultados
# ------------------------------------------------------------

print("=" * 80)
print("PASO 36 - AUDITORÍA DE ESTRUCTURA TEMPORAL PARA DEEP LEARNING")
print("=" * 80)

print("\nTOTAL DE PREDICTORES NUMÉRICOS:")
print(len(predictores_numericos))

print("\nSENTINEL-2:")
print(f"Número de variables: {len(variables_sentinel)}")
for v in variables_sentinel:
    print(" -", v)

print("\nERA5-LAND:")
print(f"Número de variables: {len(variables_era5)}")
for v in variables_era5:
    print(" -", v)

print("\nSUELO - SOILGRIDS:")
print(f"Número de variables: {len(variables_suelo)}")
for v in variables_suelo:
    print(" -", v)

print("\nESTRUCTURA AGRÍCOLA - EUROCROPS:")
print(f"Número de variables: {len(variables_eurocrops)}")
for v in variables_eurocrops:
    print(" -", v)

print("\nVARIABLES CON INFORMACIÓN TEMPORAL INTRAANUAL:")
print(f"Número de variables: {len(variables_temporales)}")
for v in variables_temporales:
    print(" -", v)

print("\nVARIABLES AGREGADAS DE CAMPAÑA:")
print(f"Número de variables: {len(variables_campana)}")
for v in variables_campana:
    print(" -", v)

print("\n" + "=" * 80)
print("RESUMEN")
print("=" * 80)

print(f"""
Predictores numéricos totales : {len(predictores_numericos)}
Sentinel-2                  : {len(variables_sentinel)}
ERA5-Land                   : {len(variables_era5)}
SoilGrids                   : {len(variables_suelo)}
EuroCrops                   : {len(variables_eurocrops)}
Variables temporales        : {len(variables_temporales)}
Variables de campaña        : {len(variables_campana)}
""")

print("PASO 36 FINALIZADO")

In [ ]:

# ============================================================
# PASO 37 - CONSTRUCCIÓN DE LAS SECUENCIAS TEMPORALES
#           PARA DEEP LEARNING
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. ERA5-Land
#    5 periodos temporales × 4 variables
# ------------------------------------------------------------

era5_periodos = [
    "ene_feb",
    "mar_abr",
    "may_jun",
    "jul_ago",
    "sep_dic"
]

era5_variables = [
    "temp_media_c",
    "temp_rocio_c",
    "precipitacion_mm",
    "radiacion_mj_m2"
]

columnas_era5_secuencia = [
    [f"{variable}_{periodo}" for variable in era5_variables]
    for periodo in era5_periodos
]

# ------------------------------------------------------------
# 2. Sentinel-2
#    5 periodos temporales × 3 índices
# ------------------------------------------------------------

sentinel_periodos = [
    "jan_feb",
    "mar_apr",
    "may_jun",
    "jul",
    "nov_dec"
]

sentinel_indices = [
    "NDVI",
    "EVI",
    "NDMI"
]

columnas_sentinel_secuencia = [
    [f"{indice}_{periodo}" for indice in sentinel_indices]
    for periodo in sentinel_periodos
]

# ------------------------------------------------------------
# 3. Comprobar existencia de todas las columnas
# ------------------------------------------------------------

cols_era5_flat = [
    c
    for periodo in columnas_era5_secuencia
    for c in periodo
]

cols_sentinel_flat = [
    c
    for periodo in columnas_sentinel_secuencia
    for c in periodo
]

faltan_era5 = [
    c for c in cols_era5_flat
    if c not in df.columns
]

faltan_sentinel = [
    c for c in cols_sentinel_flat
    if c not in df.columns
]

# ------------------------------------------------------------
# 4. Construir tensores
# ------------------------------------------------------------

if len(faltan_era5) == 0:
    X_era5_seq = np.stack(
        [
            df[cols_periodo].to_numpy(dtype=float)
            for cols_periodo in columnas_era5_secuencia
        ],
        axis=1
    )
else:
    X_era5_seq = None

if len(faltan_sentinel) == 0:
    X_sentinel_seq = np.stack(
        [
            df[cols_periodo].to_numpy(dtype=float)
            for cols_periodo in columnas_sentinel_secuencia
        ],
        axis=1
    )
else:
    X_sentinel_seq = None

# ------------------------------------------------------------
# 5. Resultados
# ------------------------------------------------------------

print("=" * 80)
print("PASO 37 - CONSTRUCCIÓN DE SECUENCIAS TEMPORALES")
print("=" * 80)

print("\nERA5-LAND")
print("Periodos:", era5_periodos)
print("Variables por periodo:", era5_variables)
print("Columnas ausentes:", faltan_era5)

if X_era5_seq is not None:
    print("Forma del tensor:", X_era5_seq.shape)
    print("NaN totales:", np.isnan(X_era5_seq).sum())

print("\nSENTINEL-2")
print("Periodos:", sentinel_periodos)
print("Índices por periodo:", sentinel_indices)
print("Columnas ausentes:", faltan_sentinel)

if X_sentinel_seq is not None:
    print("Forma del tensor:", X_sentinel_seq.shape)
    print("NaN totales:", np.isnan(X_sentinel_seq).sum())

print("\nFORMA ESPERADA:")
print(
    f"ERA5-Land   : ({len(df)}, 5, 4)"
)
print(
    f"Sentinel-2  : ({len(df)}, 5, 3)"
)

if (
    X_era5_seq is not None
    and X_sentinel_seq is not None
):
    assert X_era5_seq.shape == (len(df), 5, 4)
    assert X_sentinel_seq.shape == (len(df), 5, 3)

    print("\nESTRUCTURA TEMPORAL CONSTRUIDA CORRECTAMENTE")

print("\nPASO 37 FINALIZADO")


In [ ]:
# ============================================================
# PASO 38 - PREPARACIÓN DE DATOS PARA DEEP LEARNING
#           SIN DATA LEAKAGE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. SEPARACIÓN TEMPORAL
# ============================================================

mask_train = df["ano"] <= 2022
mask_test  = df["ano"] == 2023

print("=" * 80)
print("PASO 38 - PREPARACIÓN DE DATOS PARA DEEP LEARNING")
print("=" * 80)

print("\nTRAIN / DESARROLLO:")
print("Años:", sorted(df.loc[mask_train, "ano"].unique()))
print("Observaciones:", mask_train.sum())

print("\nTEST INDEPENDIENTE:")
print("Años:", sorted(df.loc[mask_test, "ano"].unique()))
print("Observaciones:", mask_test.sum())


# ============================================================
# 2. VARIABLE OBJETIVO
# ============================================================

y_train_dl = (
    df.loc[mask_train, "rendimiento_kg_ha"]
    .to_numpy(dtype=np.float32)
)

y_test_dl = (
    df.loc[mask_test, "rendimiento_kg_ha"]
    .to_numpy(dtype=np.float32)
)


# ============================================================
# 3. RAMA TEMPORAL ERA5-LAND
#    Forma original: n × 5 × 4
# ============================================================

X_era5_train = X_era5_seq[mask_train].astype(np.float32)
X_era5_test  = X_era5_seq[mask_test].astype(np.float32)

# Para escalar:
# convertimos temporalmente a matriz 2D
# (n*5) × 4

era5_train_2d = X_era5_train.reshape(-1, 4)
era5_test_2d  = X_era5_test.reshape(-1, 4)

scaler_era5 = StandardScaler()

# IMPORTANTE:
# ajuste SOLO con 2017-2022
era5_train_scaled = scaler_era5.fit_transform(
    era5_train_2d
)

era5_test_scaled = scaler_era5.transform(
    era5_test_2d
)

# Recuperar estructura secuencial

X_era5_train_scaled = (
    era5_train_scaled
    .reshape(
        X_era5_train.shape
    )
    .astype(np.float32)
)

X_era5_test_scaled = (
    era5_test_scaled
    .reshape(
        X_era5_test.shape
    )
    .astype(np.float32)
)


# ============================================================
# 4. RAMA TEMPORAL SENTINEL-2
#    Forma original: n × 5 × 3
# ============================================================

X_sentinel_train = (
    X_sentinel_seq[mask_train]
    .astype(np.float32)
)

X_sentinel_test = (
    X_sentinel_seq[mask_test]
    .astype(np.float32)
)

# Convertir a 2D:
# (n*5) × 3

sentinel_train_2d = (
    X_sentinel_train
    .reshape(-1, 3)
)

sentinel_test_2d = (
    X_sentinel_test
    .reshape(-1, 3)
)


# ------------------------------------------------------------
# Imputación
# ------------------------------------------------------------

imputer_sentinel = SimpleImputer(
    strategy="median"
)

# IMPORTANTE:
# mediana calculada SOLO con 2017-2022

sentinel_train_imp = (
    imputer_sentinel
    .fit_transform(
        sentinel_train_2d
    )
)

sentinel_test_imp = (
    imputer_sentinel
    .transform(
        sentinel_test_2d
    )
)


# ------------------------------------------------------------
# Escalado
# ------------------------------------------------------------

scaler_sentinel = StandardScaler()

sentinel_train_scaled = (
    scaler_sentinel
    .fit_transform(
        sentinel_train_imp
    )
)

sentinel_test_scaled = (
    scaler_sentinel
    .transform(
        sentinel_test_imp
    )
)

X_sentinel_train_scaled = (
    sentinel_train_scaled
    .reshape(
        X_sentinel_train.shape
    )
    .astype(np.float32)
)

X_sentinel_test_scaled = (
    sentinel_test_scaled
    .reshape(
        X_sentinel_test.shape
    )
    .astype(np.float32)
)


# ============================================================
# 5. RAMA ESTÁTICA
# ============================================================

variables_estaticas = [

    # SoilGrids
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm",

    # EuroCrops
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha",

    # Resumen meteorológico de campaña
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]


X_static = (
    df[variables_estaticas]
    .to_numpy(dtype=np.float32)
)

X_static_train = X_static[mask_train]
X_static_test  = X_static[mask_test]


# ------------------------------------------------------------
# Imputación variables estáticas
# ------------------------------------------------------------

imputer_static = SimpleImputer(
    strategy="median"
)

X_static_train_imp = (
    imputer_static
    .fit_transform(
        X_static_train
    )
)

X_static_test_imp = (
    imputer_static
    .transform(
        X_static_test
    )
)


# ------------------------------------------------------------
# Escalado variables estáticas
# ------------------------------------------------------------

scaler_static = StandardScaler()

X_static_train_scaled = (
    scaler_static
    .fit_transform(
        X_static_train_imp
    )
    .astype(np.float32)
)

X_static_test_scaled = (
    scaler_static
    .transform(
        X_static_test_imp
    )
    .astype(np.float32)
)


# ============================================================
# 6. CULTIVO COMO VARIABLE BINARIA
# ============================================================

# Cebada = 0
# Trigo blando = 1

mapa_cultivo = {
    "barley": 0.0,
    "common_soft_wheat": 1.0
}

X_crop = (
    df["cultivo"]
    .map(mapa_cultivo)
    .to_numpy(dtype=np.float32)
    .reshape(-1, 1)
)

X_crop_train = X_crop[mask_train]
X_crop_test  = X_crop[mask_test]


# ============================================================
# 7. CONTROLES
# ============================================================

print("\n" + "=" * 80)
print("FORMAS FINALES")
print("=" * 80)

print("\nERA5-Land")
print("TRAIN:", X_era5_train_scaled.shape)
print("TEST :", X_era5_test_scaled.shape)

print("\nSentinel-2")
print("TRAIN:", X_sentinel_train_scaled.shape)
print("TEST :", X_sentinel_test_scaled.shape)

print("\nVariables estáticas")
print("TRAIN:", X_static_train_scaled.shape)
print("TEST :", X_static_test_scaled.shape)

print("\nCultivo")
print("TRAIN:", X_crop_train.shape)
print("TEST :", X_crop_test.shape)

print("\nTarget")
print("TRAIN:", y_train_dl.shape)
print("TEST :", y_test_dl.shape)


# ============================================================
# 8. CONTROL DE NaN
# ============================================================

print("\n" + "=" * 80)
print("CONTROL DE NaN DESPUÉS DEL PREPROCESAMIENTO")
print("=" * 80)

print(
    "ERA5 train:",
    np.isnan(X_era5_train_scaled).sum()
)

print(
    "ERA5 test :",
    np.isnan(X_era5_test_scaled).sum()
)

print(
    "Sentinel train:",
    np.isnan(X_sentinel_train_scaled).sum()
)

print(
    "Sentinel test :",
    np.isnan(X_sentinel_test_scaled).sum()
)

print(
    "Static train:",
    np.isnan(X_static_train_scaled).sum()
)

print(
    "Static test :",
    np.isnan(X_static_test_scaled).sum()
)

print(
    "Target train:",
    np.isnan(y_train_dl).sum()
)

print(
    "Target test :",
    np.isnan(y_test_dl).sum()
)


# ============================================================
# 9. ASSERTS
# ============================================================

assert X_era5_train_scaled.shape == (572, 5, 4)
assert X_era5_test_scaled.shape == (95, 5, 4)

assert X_sentinel_train_scaled.shape == (572, 5, 3)
assert X_sentinel_test_scaled.shape == (95, 5, 3)

assert X_static_train_scaled.shape == (572, 12)
assert X_static_test_scaled.shape == (95, 12)

assert X_crop_train.shape == (572, 1)
assert X_crop_test.shape == (95, 1)

assert np.isnan(X_era5_train_scaled).sum() == 0
assert np.isnan(X_era5_test_scaled).sum() == 0

assert np.isnan(X_sentinel_train_scaled).sum() == 0
assert np.isnan(X_sentinel_test_scaled).sum() == 0

assert np.isnan(X_static_train_scaled).sum() == 0
assert np.isnan(X_static_test_scaled).sum() == 0


print("\nPREPROCESAMIENTO DEEP LEARNING CORRECTO")
print("2023 NO utilizado para ajustar imputadores ni escaladores.")

print("\nPASO 38 FINALIZADO")

In [ ]:
# ============================================================
# PASO 39 - ARQUITECTURA DEEP LEARNING MULTIRRAMA
# ============================================================

import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    BatchNormalization
)
from tensorflow.keras.regularizers import l2


# ============================================================
# 1. FIJAR SEMILLA PARA REPRODUCIBILIDAD
# ============================================================

SEED_DL = 42

tf.keras.utils.set_random_seed(SEED_DL)


# ============================================================
# 2. FUNCIÓN PARA CREAR EL MODELO
# ============================================================

def crear_modelo_dl_multirrama(
    lstm_era5_units=16,
    lstm_sentinel_units=12,
    dense_static_units=16,
    dense_fusion_units=24,
    dropout_rate=0.25,
    l2_reg=1e-4,
    learning_rate=1e-3
):
    """
    Arquitectura Deep Learning multirrama para integrar:

    1. Serie temporal ERA5-Land:
       5 periodos × 4 variables.

    2. Serie temporal Sentinel-2:
       5 periodos × 3 índices.

    3. Variables estáticas:
       SoilGrids + EuroCrops + agregados de campaña.

    4. Cultivo:
       cebada / trigo blando.

    Salida:
       rendimiento predicho en kg/ha.
    """

    # --------------------------------------------------------
    # RAMA 1 - ERA5-LAND
    # --------------------------------------------------------

    input_era5 = Input(
        shape=(5, 4),
        name="ERA5_Land"
    )

    x_era5 = LSTM(
        units=lstm_era5_units,
        kernel_regularizer=l2(l2_reg),
        recurrent_regularizer=l2(l2_reg),
        name="LSTM_ERA5"
    )(input_era5)

    x_era5 = Dropout(
        dropout_rate,
        name="Dropout_ERA5"
    )(x_era5)


    # --------------------------------------------------------
    # RAMA 2 - SENTINEL-2
    # --------------------------------------------------------

    input_sentinel = Input(
        shape=(5, 3),
        name="Sentinel_2"
    )

    x_sentinel = LSTM(
        units=lstm_sentinel_units,
        kernel_regularizer=l2(l2_reg),
        recurrent_regularizer=l2(l2_reg),
        name="LSTM_Sentinel"
    )(input_sentinel)

    x_sentinel = Dropout(
        dropout_rate,
        name="Dropout_Sentinel"
    )(x_sentinel)


    # --------------------------------------------------------
    # RAMA 3 - VARIABLES ESTÁTICAS
    # --------------------------------------------------------

    input_static = Input(
        shape=(12,),
        name="Variables_estaticas"
    )

    x_static = Dense(
        dense_static_units,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="Dense_Static"
    )(input_static)

    x_static = BatchNormalization(
        name="BatchNorm_Static"
    )(x_static)

    x_static = Dropout(
        dropout_rate,
        name="Dropout_Static"
    )(x_static)


    # --------------------------------------------------------
    # RAMA 4 - CULTIVO
    # --------------------------------------------------------

    input_crop = Input(
        shape=(1,),
        name="Cultivo"
    )

    x_crop = Dense(
        4,
        activation="relu",
        name="Dense_Crop"
    )(input_crop)


    # --------------------------------------------------------
    # FUSIÓN DE LAS CUATRO RAMAS
    # --------------------------------------------------------

    fusion = Concatenate(
        name="Fusion_multifuente"
    )([
        x_era5,
        x_sentinel,
        x_static,
        x_crop
    ])

    x = Dense(
        dense_fusion_units,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="Dense_Fusion_1"
    )(fusion)

    x = BatchNormalization(
        name="BatchNorm_Fusion"
    )(x)

    x = Dropout(
        dropout_rate,
        name="Dropout_Fusion"
    )(x)

    x = Dense(
        8,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="Dense_Fusion_2"
    )(x)


    # --------------------------------------------------------
    # SALIDA DE REGRESIÓN
    # --------------------------------------------------------

    output = Dense(
        1,
        activation="linear",
        name="Rendimiento_kg_ha"
    )(x)


    # --------------------------------------------------------
    # CREAR MODELO
    # --------------------------------------------------------

    model = Model(
        inputs=[
            input_era5,
            input_sentinel,
            input_static,
            input_crop
        ],
        outputs=output,
        name="DL_multirrama_rendimiento"
    )


    # --------------------------------------------------------
    # COMPILACIÓN
    # --------------------------------------------------------

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


# ============================================================
# 3. CREAR MODELO BASE
# ============================================================

modelo_dl = crear_modelo_dl_multirrama()


# ============================================================
# 4. MOSTRAR ARQUITECTURA
# ============================================================

print("=" * 80)
print("PASO 39 - ARQUITECTURA DEEP LEARNING MULTIRRAMA")
print("=" * 80)

modelo_dl.summary()


# ============================================================
# 5. INFORMACIÓN GENERAL
# ============================================================

print("\n" + "=" * 80)
print("CONTROL DE ARQUITECTURA")
print("=" * 80)

print("\nEntradas:")
print("ERA5-Land       : (5, 4)")
print("Sentinel-2      : (5, 3)")
print("Variables estáticas: (12,)")
print("Cultivo         : (1,)")

print("\nSalida:")
print("Rendimiento     : 1 valor continuo (kg/ha)")

print("\nParámetros totales:")
print(modelo_dl.count_params())

print("\n2023 utilizado para definir la arquitectura: NO")
print("2023 utilizado para entrenamiento: NO")

print("\nPASO 39 FINALIZADO")

In [ ]:
# ============================================================
# PASO 39B - VERIFICACIÓN ERA5-LAND 2023 CORREGIDO
# ============================================================

era_2023 = df[df["ano"] == 2023]

control_era_2023 = era_2023[
    [
        "precipitacion_mm_campana",
        "radiacion_mj_m2_campana",
        "temp_media_c_campana"
    ]
].mean()

print("=" * 80)
print("PASO 39B - VERIFICACIÓN ERA5-LAND 2023 CORREGIDO")
print("=" * 80)

print("\nMedias ERA5-Land 2023 presentes en el dataset maestro:")
print(control_era_2023)

print("\nValores de control esperados aproximadamente:")
print("Precipitación campaña : 646.21 mm")
print("Radiación campaña     : 6169.69 MJ/m²")
print("Temperatura media     : 15.03 °C")

print("\nDiferencias:")

print(
    "Precipitación:",
    control_era_2023["precipitacion_mm_campana"] - 646.21
)

print(
    "Radiación:",
    control_era_2023["radiacion_mj_m2_campana"] - 6169.69
)

print(
    "Temperatura:",
    control_era_2023["temp_media_c_campana"] - 15.03
)

print("\nPASO 39B FINALIZADO")

In [ ]:
# ============================================================
# PASO 40 - VALIDACIÓN TEMPORAL DEEP LEARNING
#           2020-2022
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

SEED_DL = 42

tf.keras.utils.set_random_seed(SEED_DL)

anos_validacion_dl = [
    2020,
    2021,
    2022
]

resultados_dl_folds = []


# ============================================================
# 2. FUNCIÓN DE PREPROCESAMIENTO POR FOLD
# ============================================================

def preparar_fold_dl(df, ano_validacion):
    """
    Prepara train y validación para un año concreto.

    El preprocesamiento se ajusta EXCLUSIVAMENTE con
    los años anteriores al año de validación.
    """

    mask_train_fold = (
        df["ano"] < ano_validacion
    )

    mask_val_fold = (
        df["ano"] == ano_validacion
    )


    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    y_train_fold = (
        df.loc[
            mask_train_fold,
            "rendimiento_kg_ha"
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    y_val_fold = (
        df.loc[
            mask_val_fold,
            "rendimiento_kg_ha"
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # ERA5-Land
    # --------------------------------------------------------

    X_era5_train_fold = (
        X_era5_seq[
            mask_train_fold
        ]
        .astype(
            np.float32
        )
    )

    X_era5_val_fold = (
        X_era5_seq[
            mask_val_fold
        ]
        .astype(
            np.float32
        )
    )

    era5_train_2d = (
        X_era5_train_fold
        .reshape(
            -1,
            4
        )
    )

    era5_val_2d = (
        X_era5_val_fold
        .reshape(
            -1,
            4
        )
    )

    scaler_era5_fold = (
        StandardScaler()
    )

    era5_train_scaled = (
        scaler_era5_fold
        .fit_transform(
            era5_train_2d
        )
    )

    era5_val_scaled = (
        scaler_era5_fold
        .transform(
            era5_val_2d
        )
    )

    X_era5_train_fold = (
        era5_train_scaled
        .reshape(
            X_era5_train_fold.shape
        )
        .astype(
            np.float32
        )
    )

    X_era5_val_fold = (
        era5_val_scaled
        .reshape(
            X_era5_val_fold.shape
        )
        .astype(
            np.float32
        )
    )


    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    X_sentinel_train_fold = (
        X_sentinel_seq[
            mask_train_fold
        ]
        .astype(
            np.float32
        )
    )

    X_sentinel_val_fold = (
        X_sentinel_seq[
            mask_val_fold
        ]
        .astype(
            np.float32
        )
    )

    sentinel_train_2d = (
        X_sentinel_train_fold
        .reshape(
            -1,
            3
        )
    )

    sentinel_val_2d = (
        X_sentinel_val_fold
        .reshape(
            -1,
            3
        )
    )

    imputer_sentinel_fold = (
        SimpleImputer(
            strategy="median"
        )
    )

    sentinel_train_imp = (
        imputer_sentinel_fold
        .fit_transform(
            sentinel_train_2d
        )
    )

    sentinel_val_imp = (
        imputer_sentinel_fold
        .transform(
            sentinel_val_2d
        )
    )

    scaler_sentinel_fold = (
        StandardScaler()
    )

    sentinel_train_scaled = (
        scaler_sentinel_fold
        .fit_transform(
            sentinel_train_imp
        )
    )

    sentinel_val_scaled = (
        scaler_sentinel_fold
        .transform(
            sentinel_val_imp
        )
    )

    X_sentinel_train_fold = (
        sentinel_train_scaled
        .reshape(
            X_sentinel_train_fold.shape
        )
        .astype(
            np.float32
        )
    )

    X_sentinel_val_fold = (
        sentinel_val_scaled
        .reshape(
            X_sentinel_val_fold.shape
        )
        .astype(
            np.float32
        )
    )


    # --------------------------------------------------------
    # Variables estáticas
    # --------------------------------------------------------

    X_static_full = (
        df[
            variables_estaticas
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    X_static_train_fold = (
        X_static_full[
            mask_train_fold
        ]
    )

    X_static_val_fold = (
        X_static_full[
            mask_val_fold
        ]
    )

    imputer_static_fold = (
        SimpleImputer(
            strategy="median"
        )
    )

    X_static_train_imp = (
        imputer_static_fold
        .fit_transform(
            X_static_train_fold
        )
    )

    X_static_val_imp = (
        imputer_static_fold
        .transform(
            X_static_val_fold
        )
    )

    scaler_static_fold = (
        StandardScaler()
    )

    X_static_train_fold = (
        scaler_static_fold
        .fit_transform(
            X_static_train_imp
        )
        .astype(
            np.float32
        )
    )

    X_static_val_fold = (
        scaler_static_fold
        .transform(
            X_static_val_imp
        )
        .astype(
            np.float32
        )
    )


    # --------------------------------------------------------
    # Cultivo
    # --------------------------------------------------------

    X_crop_full = (
        df["cultivo"]
        .map(
            mapa_cultivo
        )
        .to_numpy(
            dtype=np.float32
        )
        .reshape(
            -1,
            1
        )
    )

    X_crop_train_fold = (
        X_crop_full[
            mask_train_fold
        ]
    )

    X_crop_val_fold = (
        X_crop_full[
            mask_val_fold
        ]
    )


    return (
        X_era5_train_fold,
        X_sentinel_train_fold,
        X_static_train_fold,
        X_crop_train_fold,
        y_train_fold,

        X_era5_val_fold,
        X_sentinel_val_fold,
        X_static_val_fold,
        X_crop_val_fold,
        y_val_fold
    )


# ============================================================
# 3. VALIDACIÓN TEMPORAL
# ============================================================

print("=" * 80)
print("PASO 40 - VALIDACIÓN TEMPORAL DEEP LEARNING")
print("=" * 80)

for ano_val in anos_validacion_dl:

    print(
        "\n" + "=" * 80
    )

    print(
        f"VALIDACIÓN {ano_val}"
    )

    print(
        "=" * 80
    )

    datos_fold = (
        preparar_fold_dl(
            df,
            ano_val
        )
    )

    (
        X_era5_train_fold,
        X_sentinel_train_fold,
        X_static_train_fold,
        X_crop_train_fold,
        y_train_fold,

        X_era5_val_fold,
        X_sentinel_val_fold,
        X_static_val_fold,
        X_crop_val_fold,
        y_val_fold

    ) = datos_fold


    # --------------------------------------------------------
    # Crear modelo NUEVO en cada fold
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(
        SEED_DL
    )

    modelo_fold = (
        crear_modelo_dl_multirrama()
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    early_stopping = (
        EarlyStopping(
            monitor="val_loss",
            patience=25,
            restore_best_weights=True,
            min_delta=1.0
        )
    )


    # --------------------------------------------------------
    # Entrenamiento
    # --------------------------------------------------------

    history = (
        modelo_fold.fit(

            [
                X_era5_train_fold,
                X_sentinel_train_fold,
                X_static_train_fold,
                X_crop_train_fold
            ],

            y_train_fold,

            validation_data=(

                [
                    X_era5_val_fold,
                    X_sentinel_val_fold,
                    X_static_val_fold,
                    X_crop_val_fold
                ],

                y_val_fold
            ),

            epochs=300,

            batch_size=32,

            callbacks=[
                early_stopping
            ],

            verbose=0
        )
    )


    # --------------------------------------------------------
    # Predicción
    # --------------------------------------------------------

    y_pred_fold = (
        modelo_fold.predict(
            [
                X_era5_val_fold,
                X_sentinel_val_fold,
                X_static_val_fold,
                X_crop_val_fold
            ],
            verbose=0
        )
        .reshape(-1)
    )


    # --------------------------------------------------------
    # Métricas
    # --------------------------------------------------------

    mae_fold = (
        mean_absolute_error(
            y_val_fold,
            y_pred_fold
        )
    )

    rmse_fold = np.sqrt(
        mean_squared_error(
            y_val_fold,
            y_pred_fold
        )
    )

    r2_fold = (
        r2_score(
            y_val_fold,
            y_pred_fold
        )
    )

    mejor_epoch = (
        np.argmin(
            history.history[
                "val_loss"
            ]
        )
        + 1
    )


    # --------------------------------------------------------
    # Guardar
    # --------------------------------------------------------

    resultados_dl_folds.append(
        {
            "modelo":
                "DL_multirrama_LSTM",

            "ano_validacion":
                ano_val,

            "train_desde":
                int(
                    df.loc[
                        df["ano"] < ano_val,
                        "ano"
                    ].min()
                ),

            "train_hasta":
                int(
                    ano_val - 1
                ),

            "n_train":
                len(
                    y_train_fold
                ),

            "n_validacion":
                len(
                    y_val_fold
                ),

            "mejor_epoch":
                mejor_epoch,

            "MAE":
                mae_fold,

            "RMSE":
                rmse_fold,

            "R2":
                r2_fold
        }
    )


    print(
        f"n_train = {len(y_train_fold)}"
    )

    print(
        f"n_val   = {len(y_val_fold)}"
    )

    print(
        f"Mejor epoch = {mejor_epoch}"
    )

    print(
        f"MAE  = {mae_fold:.2f}"
    )

    print(
        f"RMSE = {rmse_fold:.2f}"
    )

    print(
        f"R²   = {r2_fold:.3f}"
    )


# ============================================================
# 4. RESUMEN
# ============================================================

resultados_dl_folds_df = (
    pd.DataFrame(
        resultados_dl_folds
    )
)

print(
    "\n" + "=" * 80
)

print(
    "RESULTADOS POR FOLD"
)

print(
    "=" * 80
)

display(
    resultados_dl_folds_df
)


# ============================================================
# 5. RESUMEN GLOBAL
# ============================================================

resumen_dl = pd.DataFrame(
    {
        "modelo": [
            "DL_multirrama_LSTM"
        ],

        "MAE_medio": [
            resultados_dl_folds_df[
                "MAE"
            ].mean()
        ],

        "MAE_std": [
            resultados_dl_folds_df[
                "MAE"
            ].std()
        ],

        "RMSE_medio": [
            resultados_dl_folds_df[
                "RMSE"
            ].mean()
        ],

        "RMSE_std": [
            resultados_dl_folds_df[
                "RMSE"
            ].std()
        ],

        "R2_medio": [
            resultados_dl_folds_df[
                "R2"
            ].mean()
        ],

        "R2_std": [
            resultados_dl_folds_df[
                "R2"
            ].std()
        ],

        "R2_min": [
            resultados_dl_folds_df[
                "R2"
            ].min()
        ],

        "R2_max": [
            resultados_dl_folds_df[
                "R2"
            ].max()
        ]
    }
)

print(
    "\n" + "=" * 80
)

print(
    "RESUMEN GLOBAL DEEP LEARNING"
)

print(
    "=" * 80
)

display(
    resumen_dl
)


# ============================================================
# 6. CONTROL METODOLÓGICO
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CONTROL METODOLÓGICO"
)

print(
    "=" * 80
)

print(
    "Validaciones:",
    anos_validacion_dl
)

print(
    "2023 utilizado:",
    False
)

print(
    "Imputadores ajustados dentro de cada fold:",
    True
)

print(
    "Escaladores ajustados dentro de cada fold:",
    True
)

print(
    "\nTEST 2023 CONTINÚA BLOQUEADO."
)

print(
    "\nPASO 40 FINALIZADO"
)

In [ ]:
# ============================================================
# PASO 40B - AUDITORÍA DEL DATASET MAESTRO CARGADO
# ============================================================

import os
import pandas as pd

ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"

F_DEFINITIVO = (
    f"{PROC}/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

F_ERA5_DEFINITIVO = (
    f"{PROC}/"
    "ERA5Land_features_provincia_campana_2017_2023_DEFINITIVO.csv"
)

print("=" * 90)
print("PASO 40B - AUDITORÍA DE TRAZABILIDAD ERA5")
print("=" * 90)

print("\n1. ARCHIVOS")
print("Dataset definitivo existe:", os.path.exists(F_DEFINITIVO))
print("ERA5 definitivo existe:", os.path.exists(F_ERA5_DEFINITIVO))

print("\nDataset:")
print(F_DEFINITIVO)

print("\nERA5:")
print(F_ERA5_DEFINITIVO)


# ============================================================
# 2. LEER DIRECTAMENTE LOS ARCHIVOS DE DRIVE
# ============================================================

df_drive = pd.read_csv(F_DEFINITIVO)
era5_drive = pd.read_csv(F_ERA5_DEFINITIVO)


# ============================================================
# 3. CONTROL DATASET DEFINITIVO DIRECTAMENTE DESDE DRIVE
# ============================================================

cols_control = [
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana",
    "temp_media_c_campana"
]

control_dataset = (
    df_drive.loc[
        df_drive["ano"] == 2023,
        cols_control
    ]
    .mean()
)

print("\n" + "=" * 90)
print("DATASET DEFINITIVO LEÍDO DIRECTAMENTE DE DRIVE")
print("=" * 90)

print("Shape:", df_drive.shape)
print(control_dataset.round(3))


# ============================================================
# 4. CONTROL ERA5 DEFINITIVO DIRECTAMENTE DESDE DRIVE
# ============================================================

col_ano_era5 = (
    "ano_campana"
    if "ano_campana" in era5_drive.columns
    else "ano"
)

control_era5 = (
    era5_drive.loc[
        era5_drive[col_ano_era5] == 2023,
        cols_control
    ]
    .mean()
)

print("\n" + "=" * 90)
print("ERA5 DEFINITIVO LEÍDO DIRECTAMENTE DE DRIVE")
print("=" * 90)

print("Shape:", era5_drive.shape)
print(control_era5.round(3))


# ============================================================
# 5. CONTROL DEL df ACTUAL EN MEMORIA
# ============================================================

control_memoria = (
    df.loc[
        df["ano"] == 2023,
        cols_control
    ]
    .mean()
)

print("\n" + "=" * 90)
print("df ACTUAL EN MEMORIA")
print("=" * 90)

print("Shape:", df.shape)
print(control_memoria.round(3))


# ============================================================
# 6. COMPARACIÓN
# ============================================================

comparacion = pd.DataFrame({
    "dataset_drive": control_dataset,
    "ERA5_drive": control_era5,
    "df_memoria": control_memoria
})

comparacion["dataset_vs_ERA5"] = (
    comparacion["dataset_drive"]
    - comparacion["ERA5_drive"]
)

comparacion["dataset_vs_memoria"] = (
    comparacion["dataset_drive"]
    - comparacion["df_memoria"]
)

print("\n" + "=" * 90)
print("COMPARACIÓN")
print("=" * 90)

display(comparacion.round(6))

print("\nPASO 40B FINALIZADO")

In [ ]:
# ============================================================
# PASO 40C - RECARGAR DATASET MAESTRO DEFINITIVO
# ============================================================

import pandas as pd

F_DATASET_DEFINITIVO = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos/"
    "02_Datos_procesados/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

df = pd.read_csv(F_DATASET_DEFINITIVO)

print("=" * 80)
print("PASO 40C - DATASET DEFINITIVO RECARGADO")
print("=" * 80)

print("\nArchivo:")
print(F_DATASET_DEFINITIVO)

print("\nDimensiones:")
print(df.shape)

print("\nAños:")
print(sorted(df["ano"].unique()))

print("\nCultivos:")
print(df["cultivo"].value_counts())

print("\nControl ERA5 2023:")

print(
    df.loc[
        df["ano"] == 2023,
        [
            "precipitacion_mm_campana",
            "radiacion_mj_m2_campana",
            "temp_media_c_campana"
        ]
    ]
    .mean()
    .round(3)
)

print("\nPASO 40C FINALIZADO")

In [ ]:
# ============================================================
# PASO 40D - COMPARAR ERA5 USADO EN DL CON DATASET DEFINITIVO
# ============================================================

import numpy as np

# Reconstruir ERA5 directamente desde el df DEFINITIVO
X_era5_seq_definitivo = np.stack(
    [
        df[cols_periodo].to_numpy(dtype=float)
        for cols_periodo in columnas_era5_secuencia
    ],
    axis=1
)

anos = df["ano"].to_numpy()

print("=" * 80)
print("PASO 40D - COMPARACIÓN ERA5: DL ANTERIOR vs DATASET DEFINITIVO")
print("=" * 80)

resultados_comparacion = []

for ano in sorted(df["ano"].unique()):

    mask = anos == ano

    antiguo = X_era5_seq[mask]
    definitivo = X_era5_seq_definitivo[mask]

    diferencia_abs = np.abs(
        antiguo - definitivo
    )

    max_diff = np.nanmax(diferencia_abs)
    mean_diff = np.nanmean(diferencia_abs)

    n_diferentes = np.sum(
        ~np.isclose(
            antiguo,
            definitivo,
            rtol=1e-7,
            atol=1e-9,
            equal_nan=True
        )
    )

    resultados_comparacion.append(
        {
            "ano": int(ano),
            "diferencias": int(n_diferentes),
            "diferencia_media": mean_diff,
            "diferencia_maxima": max_diff
        }
    )

    print(f"\nAño {ano}")
    print(f"Valores diferentes : {n_diferentes}")
    print(f"Diferencia media   : {mean_diff:.6f}")
    print(f"Diferencia máxima  : {max_diff:.6f}")


# ============================================================
# CONTROL ESPECÍFICO 2017-2022
# ============================================================

mask_desarrollo = anos <= 2022

iguales_desarrollo = np.allclose(
    X_era5_seq[mask_desarrollo],
    X_era5_seq_definitivo[mask_desarrollo],
    rtol=1e-7,
    atol=1e-9,
    equal_nan=True
)

print("\n" + "=" * 80)
print("CONCLUSIÓN")
print("=" * 80)

print(
    "ERA5 2017-2022 idéntico:",
    iguales_desarrollo
)

if iguales_desarrollo:
    print(
        "\nLos resultados de validación temporal del PASO 40 "
        "son válidos y NO necesitan repetirse."
    )
else:
    print(
        "\nHay diferencias en 2017-2022. "
        "El PASO 40 deberá repetirse con el dataset definitivo."
    )

print("\n2023 se mantiene sin utilizar como test DL.")

print("\nPASO 40D FINALIZADO")

In [ ]:
# ============================================================
# PASO 41 - RECONSTRUCCIÓN DE INPUTS DEEP LEARNING
#           DESDE EL DATASET MAESTRO DEFINITIVO
# ============================================================

import numpy as np

print("=" * 80)
print("PASO 41 - RECONSTRUCCIÓN DE INPUTS DL DEFINITIVOS")
print("=" * 80)


# ============================================================
# 1. ERA5-LAND
# ============================================================

X_era5_seq = np.stack(
    [
        df[cols_periodo].to_numpy(dtype=np.float32)
        for cols_periodo in columnas_era5_secuencia
    ],
    axis=1
)


# ============================================================
# 2. SENTINEL-2
# ============================================================

X_sentinel_seq = np.stack(
    [
        df[cols_periodo].to_numpy(dtype=np.float32)
        for cols_periodo in columnas_sentinel_secuencia
    ],
    axis=1
)


# ============================================================
# 3. VARIABLES ESTÁTICAS
# ============================================================

X_static = (
    df[variables_estaticas]
    .to_numpy(dtype=np.float32)
)


# ============================================================
# 4. CULTIVO
# ============================================================

X_crop = (
    df["cultivo"]
    .map(mapa_cultivo)
    .to_numpy(dtype=np.float32)
    .reshape(-1, 1)
)


# ============================================================
# 5. TARGET
# ============================================================

y_dl = (
    df["rendimiento_kg_ha"]
    .to_numpy(dtype=np.float32)
)


# ============================================================
# 6. SEPARACIÓN TEMPORAL
# ============================================================

mask_train = (
    df["ano"] <= 2022
).to_numpy()

mask_test = (
    df["ano"] == 2023
).to_numpy()


# ============================================================
# 7. CONTROLES
# ============================================================

print("\nDimensiones:")

print(
    "ERA5-Land      :",
    X_era5_seq.shape
)

print(
    "Sentinel-2     :",
    X_sentinel_seq.shape
)

print(
    "Variables static:",
    X_static.shape
)

print(
    "Cultivo        :",
    X_crop.shape
)

print(
    "Target         :",
    y_dl.shape
)


print("\nNaN:")

print(
    "ERA5-Land :",
    np.isnan(X_era5_seq).sum()
)

print(
    "Sentinel-2:",
    np.isnan(X_sentinel_seq).sum()
)

print(
    "Static     :",
    np.isnan(X_static).sum()
)

print(
    "Cultivo    :",
    np.isnan(X_crop).sum()
)

print(
    "Target     :",
    np.isnan(y_dl).sum()
)


# ============================================================
# 8. CONTROL TEMPORAL
# ============================================================

print("\nObservaciones desarrollo 2017-2022:")
print(mask_train.sum())

print("\nObservaciones test independiente 2023:")
print(mask_test.sum())


# ============================================================
# 9. CONTROL ERA5 2023
# ============================================================

print("\nControl ERA5 2023 desde df definitivo:")

print(
    df.loc[
        df["ano"] == 2023,
        [
            "precipitacion_mm_campana",
            "radiacion_mj_m2_campana",
            "temp_media_c_campana"
        ]
    ]
    .mean()
    .round(3)
)


# ============================================================
# 10. ASSERTS
# ============================================================

assert X_era5_seq.shape == (667, 5, 4)
assert X_sentinel_seq.shape == (667, 5, 3)

assert mask_train.sum() == 572
assert mask_test.sum() == 95

assert np.isnan(X_era5_seq).sum() == 0
assert np.isnan(X_crop).sum() == 0
assert np.isnan(y_dl).sum() == 0


print("\n" + "=" * 80)
print("INPUTS DL RECONSTRUIDOS DESDE DATASET DEFINITIVO")
print("=" * 80)

print(
    "2017-2022: conjunto de desarrollo."
)

print(
    "2023: test independiente todavía NO utilizado."
)

print("\nPASO 41 FINALIZADO")

# Entrenamento final DeepLearning de la LSTM multirrama

In [ ]:
# ============================================================
# PASO 42 - DETERMINAR ÉPOCAS DEL MODELO DL FINAL
#           SIN UTILIZAR 2023
# ============================================================

import numpy as np
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. SEPARACIÓN INTERNA
#    Train: 2017-2021
#    Validación: 2022
# ============================================================

mask_train_int = (
    df["ano"] <= 2021
).to_numpy()

mask_val_int = (
    df["ano"] == 2022
).to_numpy()


# ============================================================
# 2. ERA5-LAND
# ============================================================

era5_train = X_era5_seq[mask_train_int].astype(np.float32)
era5_val   = X_era5_seq[mask_val_int].astype(np.float32)

scaler_era5_final = StandardScaler()

era5_train_2d = era5_train.reshape(-1, 4)
era5_val_2d   = era5_val.reshape(-1, 4)

era5_train_sc = scaler_era5_final.fit_transform(
    era5_train_2d
)

era5_val_sc = scaler_era5_final.transform(
    era5_val_2d
)

era5_train_sc = (
    era5_train_sc
    .reshape(era5_train.shape)
    .astype(np.float32)
)

era5_val_sc = (
    era5_val_sc
    .reshape(era5_val.shape)
    .astype(np.float32)
)


# ============================================================
# 3. SENTINEL-2
# ============================================================

sent_train = X_sentinel_seq[mask_train_int].astype(np.float32)
sent_val   = X_sentinel_seq[mask_val_int].astype(np.float32)

sent_train_2d = sent_train.reshape(-1, 3)
sent_val_2d   = sent_val.reshape(-1, 3)

imputer_sentinel_final = SimpleImputer(
    strategy="median"
)

sent_train_imp = imputer_sentinel_final.fit_transform(
    sent_train_2d
)

sent_val_imp = imputer_sentinel_final.transform(
    sent_val_2d
)

scaler_sentinel_final = StandardScaler()

sent_train_sc = scaler_sentinel_final.fit_transform(
    sent_train_imp
)

sent_val_sc = scaler_sentinel_final.transform(
    sent_val_imp
)

sent_train_sc = (
    sent_train_sc
    .reshape(sent_train.shape)
    .astype(np.float32)
)

sent_val_sc = (
    sent_val_sc
    .reshape(sent_val.shape)
    .astype(np.float32)
)


# ============================================================
# 4. VARIABLES ESTÁTICAS
# ============================================================

static_train = X_static[mask_train_int].astype(np.float32)
static_val   = X_static[mask_val_int].astype(np.float32)

imputer_static_final = SimpleImputer(
    strategy="median"
)

static_train_imp = imputer_static_final.fit_transform(
    static_train
)

static_val_imp = imputer_static_final.transform(
    static_val
)

scaler_static_final = StandardScaler()

static_train_sc = scaler_static_final.fit_transform(
    static_train_imp
).astype(np.float32)

static_val_sc = scaler_static_final.transform(
    static_val_imp
).astype(np.float32)


# ============================================================
# 5. CULTIVO Y TARGET
# ============================================================

crop_train = X_crop[mask_train_int]
crop_val   = X_crop[mask_val_int]

y_train_int = y_dl[mask_train_int]
y_val_int   = y_dl[mask_val_int]


# ============================================================
# 6. CREAR MODELO NUEVO
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

modelo_seleccion_epochs = crear_modelo_dl_multirrama()


# ============================================================
# 7. EARLY STOPPING
# ============================================================

early_stopping_final = EarlyStopping(
    monitor="val_loss",
    patience=25,
    restore_best_weights=True,
    min_delta=1.0
)


# ============================================================
# 8. ENTRENAMIENTO PARA DETERMINAR ÉPOCAS
# ============================================================

history_epochs = modelo_seleccion_epochs.fit(
    [
        era5_train_sc,
        sent_train_sc,
        static_train_sc,
        crop_train
    ],
    y_train_int,

    validation_data=(
        [
            era5_val_sc,
            sent_val_sc,
            static_val_sc,
            crop_val
        ],
        y_val_int
    ),

    epochs=300,
    batch_size=32,

    callbacks=[
        early_stopping_final
    ],

    verbose=0
)


# ============================================================
# 9. MEJOR ÉPOCA
# ============================================================

mejor_epoch_final = (
    np.argmin(
        history_epochs.history["val_loss"]
    )
    + 1
)


# ============================================================
# 10. RESULTADOS
# ============================================================

print("=" * 80)
print("PASO 42 - SELECCIÓN DE ÉPOCAS DEL MODELO DL FINAL")
print("=" * 80)

print("\nTrain interno:")
print("Años: 2017-2021")
print("Observaciones:", mask_train_int.sum())

print("\nValidación interna:")
print("Año: 2022")
print("Observaciones:", mask_val_int.sum())

print("\nMejor época:")
print(mejor_epoch_final)

print("\nÉpocas realmente ejecutadas:")
print(len(history_epochs.history["loss"]))

print("\n2023 utilizado:")
print(False)

print(
    "\nLa época seleccionada se utilizará para entrenar "
    "un modelo NUEVO con todo 2017-2022."
)

print("\nPASO 42 FINALIZADO")

In [ ]:
# ============================================================
# PASO 43 - ENTRENAMIENTO FINAL DEEP LEARNING
#           TRAIN 2017-2022
#           2023 TODAVÍA BLOQUEADO
# ============================================================

import os
import joblib
import numpy as np
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

SEED_DL = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED_DL)

EPOCHS_FINAL_DL = mejor_epoch_final

print("=" * 80)
print("PASO 43 - ENTRENAMIENTO FINAL DEEP LEARNING")
print("=" * 80)

print("\nÉpocas fijadas previamente:")
print(EPOCHS_FINAL_DL)


# ============================================================
# 2. MÁSCARAS
# ============================================================

mask_dev = (
    df["ano"] <= 2022
).to_numpy()

mask_test_2023 = (
    df["ano"] == 2023
).to_numpy()

print("\nDESARROLLO / TRAIN FINAL")
print(
    "Años:",
    sorted(df.loc[mask_dev, "ano"].unique())
)
print(
    "Observaciones:",
    mask_dev.sum()
)

print("\nTEST INDEPENDIENTE")
print(
    "Año:",
    sorted(df.loc[mask_test_2023, "ano"].unique())
)
print(
    "Observaciones:",
    mask_test_2023.sum()
)


# ============================================================
# 3. TARGET
# ============================================================

y_dev_dl = (
    y_dl[mask_dev]
    .astype(np.float32)
)

y_test_2023_dl = (
    y_dl[mask_test_2023]
    .astype(np.float32)
)


# ============================================================
# 4. ERA5-LAND
# ============================================================

era5_dev = (
    X_era5_seq[mask_dev]
    .astype(np.float32)
)

era5_test = (
    X_era5_seq[mask_test_2023]
    .astype(np.float32)
)

era5_dev_2d = era5_dev.reshape(-1, 4)
era5_test_2d = era5_test.reshape(-1, 4)

scaler_era5_dl_final = StandardScaler()

era5_dev_scaled = (
    scaler_era5_dl_final
    .fit_transform(
        era5_dev_2d
    )
)

era5_test_scaled = (
    scaler_era5_dl_final
    .transform(
        era5_test_2d
    )
)

X_era5_dev_final = (
    era5_dev_scaled
    .reshape(
        era5_dev.shape
    )
    .astype(np.float32)
)

X_era5_test_final = (
    era5_test_scaled
    .reshape(
        era5_test.shape
    )
    .astype(np.float32)
)


# ============================================================
# 5. SENTINEL-2
# ============================================================

sentinel_dev = (
    X_sentinel_seq[mask_dev]
    .astype(np.float32)
)

sentinel_test = (
    X_sentinel_seq[mask_test_2023]
    .astype(np.float32)
)

sentinel_dev_2d = (
    sentinel_dev
    .reshape(-1, 3)
)

sentinel_test_2d = (
    sentinel_test
    .reshape(-1, 3)
)


# ------------------------------------------------------------
# Imputación
# ------------------------------------------------------------

imputer_sentinel_dl_final = SimpleImputer(
    strategy="median"
)

sentinel_dev_imp = (
    imputer_sentinel_dl_final
    .fit_transform(
        sentinel_dev_2d
    )
)

sentinel_test_imp = (
    imputer_sentinel_dl_final
    .transform(
        sentinel_test_2d
    )
)


# ------------------------------------------------------------
# Escalado
# ------------------------------------------------------------

scaler_sentinel_dl_final = StandardScaler()

sentinel_dev_scaled = (
    scaler_sentinel_dl_final
    .fit_transform(
        sentinel_dev_imp
    )
)

sentinel_test_scaled = (
    scaler_sentinel_dl_final
    .transform(
        sentinel_test_imp
    )
)

X_sentinel_dev_final = (
    sentinel_dev_scaled
    .reshape(
        sentinel_dev.shape
    )
    .astype(np.float32)
)

X_sentinel_test_final = (
    sentinel_test_scaled
    .reshape(
        sentinel_test.shape
    )
    .astype(np.float32)
)


# ============================================================
# 6. VARIABLES ESTÁTICAS
# ============================================================

static_dev = (
    X_static[mask_dev]
    .astype(np.float32)
)

static_test = (
    X_static[mask_test_2023]
    .astype(np.float32)
)

imputer_static_dl_final = SimpleImputer(
    strategy="median"
)

static_dev_imp = (
    imputer_static_dl_final
    .fit_transform(
        static_dev
    )
)

static_test_imp = (
    imputer_static_dl_final
    .transform(
        static_test
    )
)

scaler_static_dl_final = StandardScaler()

X_static_dev_final = (
    scaler_static_dl_final
    .fit_transform(
        static_dev_imp
    )
    .astype(np.float32)
)

X_static_test_final = (
    scaler_static_dl_final
    .transform(
        static_test_imp
    )
    .astype(np.float32)
)


# ============================================================
# 7. CULTIVO
# ============================================================

X_crop_dev_final = (
    X_crop[mask_dev]
    .astype(np.float32)
)

X_crop_test_final = (
    X_crop[mask_test_2023]
    .astype(np.float32)
)


# ============================================================
# 8. CONTROL DE NaN
# ============================================================

assert (
    np.isnan(
        X_era5_dev_final
    ).sum()
    == 0
)

assert (
    np.isnan(
        X_era5_test_final
    ).sum()
    == 0
)

assert (
    np.isnan(
        X_sentinel_dev_final
    ).sum()
    == 0
)

assert (
    np.isnan(
        X_sentinel_test_final
    ).sum()
    == 0
)

assert (
    np.isnan(
        X_static_dev_final
    ).sum()
    == 0
)

assert (
    np.isnan(
        X_static_test_final
    ).sum()
    == 0
)


# ============================================================
# 9. CREAR MODELO FINAL NUEVO
# ============================================================

tf.keras.backend.clear_session()

tf.keras.utils.set_random_seed(
    SEED_DL
)

modelo_dl_final = (
    crear_modelo_dl_multirrama()
)


# ============================================================
# 10. ENTRENAMIENTO FINAL
# ============================================================

history_final_dl = (
    modelo_dl_final.fit(

        [
            X_era5_dev_final,
            X_sentinel_dev_final,
            X_static_dev_final,
            X_crop_dev_final
        ],

        y_dev_dl,

        epochs=EPOCHS_FINAL_DL,

        batch_size=32,

        shuffle=True,

        verbose=0
    )
)


# ============================================================
# 11. GUARDAR MODELO Y PREPROCESADORES
# ============================================================

ROOT = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos"
)

DL_DIR = (
    f"{ROOT}/05_Resultados/"
    "Deep_Learning"
)

os.makedirs(
    DL_DIR,
    exist_ok=True
)

F_MODELO_DL = (
    f"{DL_DIR}/"
    "modelo_DL_multirrama_LSTM_final.keras"
)

modelo_dl_final.save(
    F_MODELO_DL
)


# Guardar objetos necesarios para reproducibilidad

joblib.dump(
    scaler_era5_dl_final,
    f"{DL_DIR}/scaler_ERA5_DL.joblib"
)

joblib.dump(
    imputer_sentinel_dl_final,
    f"{DL_DIR}/imputer_Sentinel_DL.joblib"
)

joblib.dump(
    scaler_sentinel_dl_final,
    f"{DL_DIR}/scaler_Sentinel_DL.joblib"
)

joblib.dump(
    imputer_static_dl_final,
    f"{DL_DIR}/imputer_Static_DL.joblib"
)

joblib.dump(
    scaler_static_dl_final,
    f"{DL_DIR}/scaler_Static_DL.joblib"
)


# ============================================================
# 12. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("MODELO DL FINAL ENTRENADO")
print("=" * 80)

print(
    "Train:",
    sorted(
        df.loc[
            mask_dev,
            "ano"
        ].unique()
    )
)

print(
    "n train:",
    mask_dev.sum()
)

print(
    "Épocas:",
    EPOCHS_FINAL_DL
)

print(
    "Parámetros:",
    modelo_dl_final.count_params()
)

print(
    "\nModelo guardado:"
)

print(
    F_MODELO_DL
)

print(
    "\n2023 utilizado para entrenamiento:",
    False
)

print(
    "2023 utilizado para seleccionar arquitectura:",
    False
)

print(
    "2023 utilizado para elegir épocas:",
    False
)

print(
    "\nEl test 2023 está preparado mediante transform(),"
)
print(
    "pero sus predicciones todavía NO se han calculado."
)

print("\nPASO 43 FINALIZADO")

# Test independiente 2023 del Deep Learning

In [ ]:
# ============================================================
# PASO 44 - TEST INDEPENDIENTE 2023
#           DEEP LEARNING MULTIRRAMA
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. PREDICCIÓN 2023
# ============================================================

y_pred_dl_2023 = (
    modelo_dl_final.predict(
        [
            X_era5_test_final,
            X_sentinel_test_final,
            X_static_test_final,
            X_crop_test_final
        ],
        verbose=0
    )
    .reshape(-1)
)


# ============================================================
# 2. MÉTRICAS GLOBALES
# ============================================================

mae_dl_2023 = mean_absolute_error(
    y_test_2023_dl,
    y_pred_dl_2023
)

rmse_dl_2023 = np.sqrt(
    mean_squared_error(
        y_test_2023_dl,
        y_pred_dl_2023
    )
)

r2_dl_2023 = r2_score(
    y_test_2023_dl,
    y_pred_dl_2023
)

sesgo_dl_2023 = (
    y_pred_dl_2023
    - y_test_2023_dl
).mean()


# ============================================================
# 3. TABLA GLOBAL
# ============================================================

resultado_dl_2023 = pd.DataFrame(
    {
        "modelo": [
            "DL_multirrama_LSTM"
        ],

        "MAE": [
            mae_dl_2023
        ],

        "RMSE": [
            rmse_dl_2023
        ],

        "R2": [
            r2_dl_2023
        ],

        "sesgo_pred_menos_obs": [
            sesgo_dl_2023
        ]
    }
)


print("=" * 80)
print("PASO 44 - TEST INDEPENDIENTE 2023 DEEP LEARNING")
print("=" * 80)

print("\nRESULTADO GLOBAL 2023")

display(
    resultado_dl_2023.round(3)
)


# ============================================================
# 4. RESULTADOS POR CULTIVO
# ============================================================

df_test_dl = (
    df.loc[
        mask_test_2023,
        [
            "provincia_estandar",
            "cultivo",
            "rendimiento_kg_ha"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

df_test_dl["pred_DL"] = (
    y_pred_dl_2023
)

resultados_dl_cultivo = []

for cultivo in [
    "barley",
    "common_soft_wheat"
]:

    sub = df_test_dl[
        df_test_dl["cultivo"]
        == cultivo
    ]

    y_obs = sub[
        "rendimiento_kg_ha"
    ].to_numpy()

    y_pred = sub[
        "pred_DL"
    ].to_numpy()

    resultados_dl_cultivo.append(
        {
            "modelo":
                "DL_multirrama_LSTM",

            "cultivo":
                cultivo,

            "n":
                len(sub),

            "observado_medio":
                y_obs.mean(),

            "predicho_medio":
                y_pred.mean(),

            "sesgo":
                (
                    y_pred
                    - y_obs
                ).mean(),

            "MAE":
                mean_absolute_error(
                    y_obs,
                    y_pred
                ),

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        y_obs,
                        y_pred
                    )
                ),

            "R2":
                r2_score(
                    y_obs,
                    y_pred
                )
        }
    )


resultados_dl_cultivo_df = (
    pd.DataFrame(
        resultados_dl_cultivo
    )
)

print(
    "\n" + "=" * 80
)

print(
    "RESULTADOS 2023 POR CULTIVO"
)

print(
    "=" * 80
)

display(
    resultados_dl_cultivo_df.round(3)
)


# ============================================================
# 5. COMPARACIÓN CON MODELOS CLÁSICOS
# ============================================================

comparacion_modelos_2023 = pd.DataFrame(
    [
        {
            "modelo":
                "DL_multirrama_LSTM",

            "MAE":
                mae_dl_2023,

            "RMSE":
                rmse_dl_2023,

            "R2":
                r2_dl_2023,

            "sesgo_pred_menos_obs":
                sesgo_dl_2023
        },

        {
            "modelo":
                "XGB_tuned",

            "MAE":
                926.008,

            "RMSE":
                1102.817,

            "R2":
                -0.126,

            "sesgo_pred_menos_obs":
                840.343
        },

        {
            "modelo":
                "RF_XGB_50_50",

            "MAE":
                949.490,

            "RMSE":
                1118.798,

            "R2":
                -0.159,

            "sesgo_pred_menos_obs":
                882.235
        },

        {
            "modelo":
                "RF70_XGB30",

            "MAE":
                961.305,

            "RMSE":
                1129.432,

            "R2":
                -0.181,

            "sesgo_pred_menos_obs":
                898.992
        },

        {
            "modelo":
                "RF_tuned",

            "MAE":
                980.285,

            "RMSE":
                1149.732,

            "R2":
                -0.224,

            "sesgo_pred_menos_obs":
                924.127
        }
    ]
)


comparacion_modelos_2023 = (
    comparacion_modelos_2023
    .sort_values(
        by="RMSE"
    )
    .reset_index(drop=True)
)

print(
    "\n" + "=" * 80
)

print(
    "COMPARACIÓN FINAL - TEST 2023"
)

print(
    "=" * 80
)

display(
    comparacion_modelos_2023.round(3)
)


# ============================================================
# 6. VALIDACIÓN TEMPORAL vs TEST
# ============================================================

comparacion_dl_validacion_test = pd.DataFrame(
    {
        "modelo": [
            "DL_multirrama_LSTM"
        ],

        "RMSE_validacion_2020_2022": [
            resultados_dl_folds_df[
                "RMSE"
            ].mean()
        ],

        "R2_validacion_2020_2022": [
            resultados_dl_folds_df[
                "R2"
            ].mean()
        ],

        "RMSE_test_2023": [
            rmse_dl_2023
        ],

        "R2_test_2023": [
            r2_dl_2023
        ]
    }
)

print(
    "\n" + "=" * 80
)

print(
    "DEEP LEARNING - VALIDACIÓN vs TEST"
)

print(
    "=" * 80
)

display(
    comparacion_dl_validacion_test.round(3)
)


# ============================================================
# 7. GUARDAR RESULTADOS
# ============================================================

F_RESULTADOS_DL_2023 = (
    f"{DL_DIR}/"
    "paso44_test_2023_DL_multirrama.csv"
)

F_RESULTADOS_DL_CULTIVO = (
    f"{DL_DIR}/"
    "paso44_test_2023_DL_por_cultivo.csv"
)

F_COMPARACION_DL = (
    f"{DL_DIR}/"
    "paso44_comparacion_DL_vs_ML_2023.csv"
)

resultado_dl_2023.to_csv(
    F_RESULTADOS_DL_2023,
    index=False
)

resultados_dl_cultivo_df.to_csv(
    F_RESULTADOS_DL_CULTIVO,
    index=False
)

comparacion_modelos_2023.to_csv(
    F_COMPARACION_DL,
    index=False
)


# ============================================================
# 8. CONTROL FINAL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CONTROL METODOLÓGICO FINAL"
)

print(
    "=" * 80
)

print(
    "Modelo entrenado con:",
    sorted(
        df.loc[
            mask_dev,
            "ano"
        ].unique()
    )
)

print(
    "Test:",
    [2023]
)

print(
    "n train:",
    mask_dev.sum()
)

print(
    "n test:",
    mask_test_2023.sum()
)

print(
    "\n2023 se ha utilizado exclusivamente como TEST FINAL del modelo DL."
)

print(
    "No realizar tuning ni modificar la arquitectura usando estos resultados."
)

print(
    "\nPASO 44 FINALIZADO"
)

In [ ]:
# ============================================================
# PASO 45 - DIAGNÓSTICO DEL MODELO DL
#           TEST INDEPENDIENTE 2023
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. CARPETA DE RESULTADOS
# ============================================================

FIG_DIR = f"{DL_DIR}/Figuras"

os.makedirs(
    FIG_DIR,
    exist_ok=True
)


# ============================================================
# 2. TABLA DE PREDICCIONES
# ============================================================

diagnostico_dl = (
    df.loc[
        mask_test_2023,
        [
            "provincia_estandar",
            "ano",
            "cultivo",
            "rendimiento_kg_ha"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

diagnostico_dl["predicho_kg_ha"] = (
    y_pred_dl_2023
)

diagnostico_dl["residuo_kg_ha"] = (
    diagnostico_dl["rendimiento_kg_ha"]
    - diagnostico_dl["predicho_kg_ha"]
)

diagnostico_dl["error_absoluto_kg_ha"] = (
    np.abs(
        diagnostico_dl["residuo_kg_ha"]
    )
)


# ============================================================
# 3. GUARDAR PREDICCIONES
# ============================================================

F_PREDICCIONES = (
    f"{DL_DIR}/"
    "predicciones_DL_test_2023.csv"
)

diagnostico_dl.to_csv(
    F_PREDICCIONES,
    index=False
)


# ============================================================
# 4. FIGURA 1
#    OBSERVADO vs PREDICHO
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)

ax.scatter(
    diagnostico_dl["rendimiento_kg_ha"],
    diagnostico_dl["predicho_kg_ha"],
    alpha=0.70
)

lim_min = min(
    diagnostico_dl["rendimiento_kg_ha"].min(),
    diagnostico_dl["predicho_kg_ha"].min()
)

lim_max = max(
    diagnostico_dl["rendimiento_kg_ha"].max(),
    diagnostico_dl["predicho_kg_ha"].max()
)

ax.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    linestyle="--",
    linewidth=1.5,
    label="Predicción perfecta"
)

ax.set_xlabel(
    "Rendimiento observado (kg/ha)"
)

ax.set_ylabel(
    "Rendimiento predicho (kg/ha)"
)

ax.set_title(
    "Deep Learning multirrama - Test independiente 2023"
)

ax.text(
    0.05,
    0.95,
    (
        f"n = {len(diagnostico_dl)}\n"
        f"R² = {r2_dl_2023:.3f}\n"
        f"RMSE = {rmse_dl_2023:.1f} kg/ha\n"
        f"MAE = {mae_dl_2023:.1f} kg/ha"
    ),
    transform=ax.transAxes,
    verticalalignment="top"
)

ax.legend()

fig.tight_layout()

F_FIG1 = (
    f"{FIG_DIR}/"
    "DL_2023_observado_vs_predicho.png"
)

fig.savefig(
    F_FIG1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 5. FIGURA 2
#    RESIDUOS vs PREDICCIÓN
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    diagnostico_dl["predicho_kg_ha"],
    diagnostico_dl["residuo_kg_ha"],
    alpha=0.70
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel(
    "Rendimiento predicho (kg/ha)"
)

ax.set_ylabel(
    "Residuo: observado - predicho (kg/ha)"
)

ax.set_title(
    "Residuos del modelo Deep Learning - Test 2023"
)

fig.tight_layout()

F_FIG2 = (
    f"{FIG_DIR}/"
    "DL_2023_residuos_vs_prediccion.png"
)

fig.savefig(
    F_FIG2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 6. FIGURA 3
#    DISTRIBUCIÓN DE RESIDUOS
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.hist(
    diagnostico_dl["residuo_kg_ha"],
    bins=15,
    edgecolor="black",
    alpha=0.75
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.axvline(
    diagnostico_dl[
        "residuo_kg_ha"
    ].mean(),
    linestyle=":",
    linewidth=1.5,
    label=(
        "Residuo medio = "
        f"{diagnostico_dl['residuo_kg_ha'].mean():.1f} kg/ha"
    )
)

ax.set_xlabel(
    "Residuo (kg/ha)"
)

ax.set_ylabel(
    "Frecuencia"
)

ax.set_title(
    "Distribución de residuos - Deep Learning, test 2023"
)

ax.legend()

fig.tight_layout()

F_FIG3 = (
    f"{FIG_DIR}/"
    "DL_2023_distribucion_residuos.png"
)

fig.savefig(
    F_FIG3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. FIGURA 4
#    OBSERVADO vs PREDICHO POR CULTIVO
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)

for cultivo in [
    "barley",
    "common_soft_wheat"
]:

    sub = diagnostico_dl[
        diagnostico_dl["cultivo"]
        == cultivo
    ]

    etiqueta = (
        "Cebada"
        if cultivo == "barley"
        else "Trigo blando"
    )

    ax.scatter(
        sub["rendimiento_kg_ha"],
        sub["predicho_kg_ha"],
        alpha=0.75,
        label=etiqueta
    )


ax.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    linestyle="--",
    linewidth=1.5,
    label="Predicción perfecta"
)

ax.set_xlabel(
    "Rendimiento observado (kg/ha)"
)

ax.set_ylabel(
    "Rendimiento predicho (kg/ha)"
)

ax.set_title(
    "Deep Learning - Predicción 2023 por cultivo"
)

ax.legend()

fig.tight_layout()

F_FIG4 = (
    f"{FIG_DIR}/"
    "DL_2023_observado_vs_predicho_por_cultivo.png"
)

fig.savefig(
    F_FIG4,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. ESTADÍSTICOS DE RESIDUOS
# ============================================================

residuos = (
    diagnostico_dl[
        "residuo_kg_ha"
    ]
)

resumen_residuos = pd.DataFrame(
    {
        "estadistico": [
            "Media",
            "Mediana",
            "Desviación estándar",
            "Mínimo",
            "Máximo"
        ],

        "residuo_kg_ha": [
            residuos.mean(),
            residuos.median(),
            residuos.std(),
            residuos.min(),
            residuos.max()
        ]
    }
)

print("=" * 80)
print("PASO 45 - DIAGNÓSTICO DEL MODELO DL")
print("=" * 80)

print("\nRESUMEN DE RESIDUOS")
display(
    resumen_residuos.round(3)
)


# ============================================================
# 9. MAYORES ERRORES
# ============================================================

mayores_errores = (
    diagnostico_dl
    .sort_values(
        "error_absoluto_kg_ha",
        ascending=False
    )
    .head(10)
)

print(
    "\n10 OBSERVACIONES CON MAYOR ERROR ABSOLUTO"
)

display(
    mayores_errores.round(2)
)


# ============================================================
# 10. CONTROL DE ARCHIVOS
# ============================================================

print("\nArchivos guardados:")

for f in [
    F_PREDICCIONES,
    F_FIG1,
    F_FIG2,
    F_FIG3,
    F_FIG4
]:
    print(" -", f)


print("\nPASO 45 FINALIZADO")

In [ ]:
# ============================================================
# PASO 46 - INTERPRETABILIDAD DEL MODELO DEEP LEARNING
#           PERMUTATION IMPORTANCE POR BLOQUES
#           TEST INDEPENDIENTE 2023
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

N_REPETICIONES = 30
SEED_PERM = 42

rng = np.random.default_rng(SEED_PERM)

print("=" * 80)
print("PASO 46 - PERMUTATION IMPORTANCE POR BLOQUES")
print("=" * 80)

print("\nRepeticiones por bloque:", N_REPETICIONES)
print("Test independiente:", 2023)
print("Modelo modificado:", False)


# ============================================================
# 2. RMSE BASE DEL MODELO SIN ALTERAR
# ============================================================

rmse_base = np.sqrt(
    mean_squared_error(
        y_test_2023_dl,
        y_pred_dl_2023
    )
)

print(
    "\nRMSE original 2023:",
    round(rmse_base, 3),
    "kg/ha"
)


# ============================================================
# 3. FUNCIÓN DE PERMUTACIÓN
# ============================================================

def evaluar_bloque_permutado(
    nombre_bloque,
    n_repeticiones=30
):

    resultados = []

    n = len(y_test_2023_dl)

    for repeticion in range(
        n_repeticiones
    ):

        # Copias independientes
        era5_perm = (
            X_era5_test_final.copy()
        )

        sentinel_perm = (
            X_sentinel_test_final.copy()
        )

        static_perm = (
            X_static_test_final.copy()
        )

        crop_perm = (
            X_crop_test_final.copy()
        )

        # ----------------------------------------------------
        # Permutación conjunta por observaciones
        # ----------------------------------------------------

        indices_perm = (
            rng.permutation(n)
        )

        if nombre_bloque == "ERA5-Land":

            era5_perm = (
                era5_perm[
                    indices_perm
                ]
            )

        elif nombre_bloque == "Sentinel-2":

            sentinel_perm = (
                sentinel_perm[
                    indices_perm
                ]
            )

        elif nombre_bloque == "Variables estáticas":

            static_perm = (
                static_perm[
                    indices_perm
                ]
            )

        elif nombre_bloque == "Cultivo":

            crop_perm = (
                crop_perm[
                    indices_perm
                ]
            )

        else:

            raise ValueError(
                "Bloque no reconocido."
            )

        # ----------------------------------------------------
        # Predicción sin reentrenamiento
        # ----------------------------------------------------

        pred_perm = (
            modelo_dl_final.predict(
                [
                    era5_perm,
                    sentinel_perm,
                    static_perm,
                    crop_perm
                ],
                verbose=0
            )
            .reshape(-1)
        )

        rmse_perm = np.sqrt(
            mean_squared_error(
                y_test_2023_dl,
                pred_perm
            )
        )

        incremento_rmse = (
            rmse_perm
            - rmse_base
        )

        incremento_pct = (
            incremento_rmse
            / rmse_base
            * 100
        )

        resultados.append(
            {
                "bloque":
                    nombre_bloque,

                "repeticion":
                    repeticion + 1,

                "RMSE_permutado":
                    rmse_perm,

                "incremento_RMSE":
                    incremento_rmse,

                "incremento_RMSE_pct":
                    incremento_pct
            }
        )

    return resultados


# ============================================================
# 4. EJECUTAR PARA LOS CUATRO BLOQUES
# ============================================================

bloques = [
    "ERA5-Land",
    "Sentinel-2",
    "Variables estáticas",
    "Cultivo"
]

resultados_perm = []

for bloque in bloques:

    print(
        "\nEvaluando:",
        bloque
    )

    resultados_perm.extend(
        evaluar_bloque_permutado(
            bloque,
            N_REPETICIONES
        )
    )


resultados_perm_df = pd.DataFrame(
    resultados_perm
)


# ============================================================
# 5. RESUMEN
# ============================================================

resumen_perm = (
    resultados_perm_df
    .groupby("bloque")
    .agg(
        RMSE_permutado_medio=(
            "RMSE_permutado",
            "mean"
        ),

        RMSE_permutado_std=(
            "RMSE_permutado",
            "std"
        ),

        incremento_RMSE_medio=(
            "incremento_RMSE",
            "mean"
        ),

        incremento_RMSE_std=(
            "incremento_RMSE",
            "std"
        ),

        incremento_RMSE_pct=(
            "incremento_RMSE_pct",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "incremento_RMSE_medio",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n" + "=" * 80
)

print(
    "IMPORTANCIA POR BLOQUES"
)

print(
    "=" * 80
)

display(
    resumen_perm.round(3)
)


# ============================================================
# 6. FIGURA
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    resumen_perm["bloque"],
    resumen_perm[
        "incremento_RMSE_medio"
    ],
    yerr=resumen_perm[
        "incremento_RMSE_std"
    ],
    capsize=4
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_ylabel(
    "Incremento del RMSE (kg/ha)"
)

ax.set_xlabel(
    "Bloque de variables"
)

ax.set_title(
    "Importancia por permutación de las fuentes de información"
)

plt.xticks(
    rotation=20,
    ha="right"
)

fig.tight_layout()

F_FIG_PERM = (
    f"{FIG_DIR}/"
    "DL_2023_permutation_importance_bloques.png"
)

fig.savefig(
    F_FIG_PERM,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. GUARDAR RESULTADOS
# ============================================================

F_PERM_DETALLE = (
    f"{DL_DIR}/"
    "paso46_permutation_importance_detalle.csv"
)

F_PERM_RESUMEN = (
    f"{DL_DIR}/"
    "paso46_permutation_importance_resumen.csv"
)

resultados_perm_df.to_csv(
    F_PERM_DETALLE,
    index=False
)

resumen_perm.to_csv(
    F_PERM_RESUMEN,
    index=False
)


# ============================================================
# 8. CONTROL FINAL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CONTROL METODOLÓGICO"
)

print(
    "=" * 80
)

print(
    "Modelo reentrenado:",
    False
)

print(
    "Arquitectura modificada:",
    False
)

print(
    "Variables seleccionadas usando 2023:",
    False
)

print(
    "Interpretación realizada post hoc:",
    True
)

print(
    "\nLos resultados NO deben utilizarse para modificar el modelo."
)

print(
    "\nArchivos guardados:"
)

print(
    F_PERM_DETALLE
)

print(
    F_PERM_RESUMEN
)

print(
    F_FIG_PERM
)

print(
    "\nPASO 46 FINALIZADO"
)

In [ ]:
# ============================================================
# PASO 47 - IMPORTANCIA TEMPORAL DEL MODELO DEEP LEARNING
#           PERMUTATION IMPORTANCE POR PERIODOS
#           ERA5-LAND Y SENTINEL-2
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

N_REPETICIONES = 30
SEED_TEMP = 42

rng_temp = np.random.default_rng(SEED_TEMP)

rmse_base = np.sqrt(
    mean_squared_error(
        y_test_2023_dl,
        y_pred_dl_2023
    )
)

periodos_era5 = [
    "ene_feb",
    "mar_abr",
    "may_jun",
    "jul_ago",
    "sep_dic"
]

periodos_sentinel = [
    "jan_feb",
    "mar_apr",
    "may_jun",
    "jul",
    "nov_dec"
]


print("=" * 80)
print("PASO 47 - IMPORTANCIA TEMPORAL")
print("=" * 80)

print(
    "\nRMSE original 2023:",
    round(rmse_base, 3),
    "kg/ha"
)


# ============================================================
# 2. FUNCIÓN DE EVALUACIÓN
# ============================================================

def evaluar_periodo(
    fuente,
    indice_periodo,
    nombre_periodo,
    n_repeticiones=30
):

    resultados = []

    n = len(y_test_2023_dl)

    for repeticion in range(
        n_repeticiones
    ):

        era5_perm = (
            X_era5_test_final.copy()
        )

        sentinel_perm = (
            X_sentinel_test_final.copy()
        )

        indices_perm = (
            rng_temp.permutation(n)
        )

        # ----------------------------------------------------
        # Permutamos TODAS las variables del periodo
        # manteniendo unido el vector de dicho periodo
        # ----------------------------------------------------

        if fuente == "ERA5-Land":

            era5_perm[
                :,
                indice_periodo,
                :
            ] = (
                era5_perm[
                    indices_perm,
                    indice_periodo,
                    :
                ]
            )

        elif fuente == "Sentinel-2":

            sentinel_perm[
                :,
                indice_periodo,
                :
            ] = (
                sentinel_perm[
                    indices_perm,
                    indice_periodo,
                    :
                ]
            )

        else:

            raise ValueError(
                "Fuente no reconocida."
            )

        pred_perm = (
            modelo_dl_final.predict(
                [
                    era5_perm,
                    sentinel_perm,
                    X_static_test_final,
                    X_crop_test_final
                ],
                verbose=0
            )
            .reshape(-1)
        )

        rmse_perm = np.sqrt(
            mean_squared_error(
                y_test_2023_dl,
                pred_perm
            )
        )

        incremento = (
            rmse_perm
            - rmse_base
        )

        resultados.append(
            {
                "fuente":
                    fuente,

                "periodo":
                    nombre_periodo,

                "repeticion":
                    repeticion + 1,

                "RMSE_permutado":
                    rmse_perm,

                "incremento_RMSE":
                    incremento,

                "incremento_RMSE_pct":
                    incremento
                    / rmse_base
                    * 100
            }
        )

    return resultados


# ============================================================
# 3. ERA5-LAND
# ============================================================

resultados_temporales = []

for i, periodo in enumerate(
    periodos_era5
):

    print(
        "Evaluando ERA5-Land:",
        periodo
    )

    resultados_temporales.extend(
        evaluar_periodo(
            fuente="ERA5-Land",
            indice_periodo=i,
            nombre_periodo=periodo,
            n_repeticiones=N_REPETICIONES
        )
    )


# ============================================================
# 4. SENTINEL-2
# ============================================================

for i, periodo in enumerate(
    periodos_sentinel
):

    print(
        "Evaluando Sentinel-2:",
        periodo
    )

    resultados_temporales.extend(
        evaluar_periodo(
            fuente="Sentinel-2",
            indice_periodo=i,
            nombre_periodo=periodo,
            n_repeticiones=N_REPETICIONES
        )
    )


resultados_temporales_df = (
    pd.DataFrame(
        resultados_temporales
    )
)


# ============================================================
# 5. RESUMEN
# ============================================================

resumen_temporal = (
    resultados_temporales_df
    .groupby(
        [
            "fuente",
            "periodo"
        ]
    )
    .agg(
        RMSE_permutado_medio=(
            "RMSE_permutado",
            "mean"
        ),

        RMSE_permutado_std=(
            "RMSE_permutado",
            "std"
        ),

        incremento_RMSE_medio=(
            "incremento_RMSE",
            "mean"
        ),

        incremento_RMSE_std=(
            "incremento_RMSE",
            "std"
        ),

        incremento_RMSE_pct=(
            "incremento_RMSE_pct",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 6. ORDENAR DENTRO DE CADA FUENTE
# ============================================================

resumen_era5 = (
    resumen_temporal[
        resumen_temporal["fuente"]
        == "ERA5-Land"
    ]
    .sort_values(
        "incremento_RMSE_medio",
        ascending=False
    )
    .reset_index(drop=True)
)

resumen_sentinel = (
    resumen_temporal[
        resumen_temporal["fuente"]
        == "Sentinel-2"
    ]
    .sort_values(
        "incremento_RMSE_medio",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n" + "=" * 80
)

print(
    "ERA5-LAND - IMPORTANCIA POR PERIODO"
)

print(
    "=" * 80
)

display(
    resumen_era5.round(3)
)


print(
    "\n" + "=" * 80
)

print(
    "SENTINEL-2 - IMPORTANCIA POR PERIODO"
)

print(
    "=" * 80
)

display(
    resumen_sentinel.round(3)
)


# ============================================================
# 7. FIGURA ERA5-LAND
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    resumen_era5["periodo"],
    resumen_era5[
        "incremento_RMSE_medio"
    ],
    yerr=resumen_era5[
        "incremento_RMSE_std"
    ],
    capsize=4
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel(
    "Periodo"
)

ax.set_ylabel(
    "Incremento del RMSE (kg/ha)"
)

ax.set_title(
    "Importancia temporal de ERA5-Land"
)

fig.tight_layout()

F_FIG_ERA5_TEMP = (
    f"{FIG_DIR}/"
    "DL_2023_importancia_temporal_ERA5.png"
)

fig.savefig(
    F_FIG_ERA5_TEMP,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 8. FIGURA SENTINEL-2
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    resumen_sentinel["periodo"],
    resumen_sentinel[
        "incremento_RMSE_medio"
    ],
    yerr=resumen_sentinel[
        "incremento_RMSE_std"
    ],
    capsize=4
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel(
    "Periodo"
)

ax.set_ylabel(
    "Incremento del RMSE (kg/ha)"
)

ax.set_title(
    "Importancia temporal de Sentinel-2"
)

fig.tight_layout()

F_FIG_SENT_TEMP = (
    f"{FIG_DIR}/"
    "DL_2023_importancia_temporal_Sentinel2.png"
)

fig.savefig(
    F_FIG_SENT_TEMP,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. GUARDAR
# ============================================================

F_TEMP_DETALLE = (
    f"{DL_DIR}/"
    "paso47_importancia_temporal_detalle.csv"
)

F_TEMP_RESUMEN = (
    f"{DL_DIR}/"
    "paso47_importancia_temporal_resumen.csv"
)

resultados_temporales_df.to_csv(
    F_TEMP_DETALLE,
    index=False
)

resumen_temporal.to_csv(
    F_TEMP_RESUMEN,
    index=False
)


# ============================================================
# 10. CONTROL METODOLÓGICO
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CONTROL METODOLÓGICO"
)

print(
    "=" * 80
)

print(
    "Modelo reentrenado:",
    False
)

print(
    "Arquitectura modificada:",
    False
)

print(
    "Selección de variables:",
    False
)

print(
    "Análisis post hoc sobre test 2023:",
    True
)

print(
    "\nPASO 47 FINALIZADO"
)